In [2]:
%load_ext autoreload
%autoreload 2
%aimport src_config

In [3]:
%%time
from degiro.degiroUtils import _MODEL_STATS_PATH_    # "C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\parquet\\"
from degiro.degiroUtils import _MODEL_PATH_          # "C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\models\\"
from degiro import logger
from degiro.degiroUtils import downloader, anagrafica_builder
from degiro.degiroUtils import check_equality_btwn_df
import degiroapi
import degiro.degiroapi_mio as dg
from pathlib import Path
import pandas as pd
import numpy as np

Wall time: 32.5 s


In [48]:
def load_models_df(raw_file_path: Path, model_code: str):
    """
    """
    models = pd.read_parquet(path=raw_file_path / "models_{}.parquet".format(model_code), engine="fastparquet").set_index("index")
    models.index.name = None
    return models

def resume_models_df(grid_df: pd.DataFrame, raw_file_path: Path):
    """
    Fct for loading the saved model parametres through the PARAM_HASH key (index of the grid_df pd.DataFrame) and concatenate them building the models_df.
    The grid_df dataframe s index is PARAM_HASH and contains params, stats and MODEL_HASH (key for ) 
    """
    models = grid_df.reset_index().apply(lambda x: load_models_df(raw_file_path, x.PARAM_HASH), axis=1)
    models_df = pd.DataFrame()
    for _, model in models.iteritems():
        models_df = pd.concat([models_df, model])

    models_df.reset_index(inplace=True)
    models_df.rename(columns={"index" : "ISIN"}, inplace=True)
    models_df.set_index(["PARAM_HASH", "ISIN"], inplace=True)
    return models_df

def train_on_grid_refactor(
    global_param: dict, 
    grid_param: dict, 
    ts: pd.DataFrame, 
    return_stats: bool=True, 
    save: bool=False, 
    export_models: bool=False
    ):
    from copy import deepcopy
    import time
    from degiro.Timer import Timer
    from degiro.degiroUtils import rnn_save
    from degiro.degiroUtils import get_param_permutation
    #from tabulate import tabulate
    
    _BEST_CHOICE_CRITERIA_ = "TEST_LOSS_PERC"

    param_permutations = get_param_permutation(grid_param)
    total_permutations = len(param_permutations)

    param_df = pd.DataFrame()
    stats_df = pd.DataFrame()
    stats_models_df = pd.DataFrame()
    logger.debug("Total iterations...{}".format(total_permutations))
    
    timer = Timer(total_iterations=total_permutations)
    logger.debug(param_permutations)

    for iteration, model_p in enumerate(param_permutations):
        iteration += 1
        start_time = time.time()
        logger.debug(model_p)
        
        global_param.update(model_p)
        # Giro con save e return_stats defaultati a False e True -> train_models_on_df_refactor non scrive nulla
        it_stats_models, param_hash = train_models_on_df_refactor(
            ts=ts, 
            model_param=deepcopy(global_param), 
            return_stats=return_stats, 
            save=save
            )

        stats = it_stats_models[["TRAIN_LOSS", "TEST_LOSS", "VOL_IS", "VOL_OOS"]].describe()
        #print(tabulate(stats, headers='keys', tablefmt='psql'))

        stats["PARAM_HASH"] = param_hash
        stats.reset_index(inplace=True)
        stats.rename(columns={"index" : "STAT"}, inplace=True)
        stats.set_index(["PARAM_HASH", "STAT"], inplace=True)
        stats_df = pd.concat([stats_df, stats])

        param_tmp = pd.DataFrame(model_p, index=[param_hash])
        param_df = pd.concat([param_df, param_tmp])
        stats_models_df = pd.concat([stats_models_df, it_stats_models])

        logger.debug("Done {} -> {}%".format(iteration, (iteration)/total_permutations*100))

        timer.get_stats(start_time=start_time, iteration=iteration)

    param_df.index.name = "PARAM_HASH"
    
    # TODO: print best performer param
    # TODO: select best param per isin
    # TODO: save only best models
    stats_models_df.reset_index(inplace=True)
    stats_models_df.rename(columns={"index" : "ISIN"}, inplace=True)

    best_models_df = stats_models_df.loc[stats_models_df.groupby("ISIN")[_BEST_CHOICE_CRITERIA_].idxmin()].sort_values("PARAM_HASH").merge(
        param_df, left_on="PARAM_HASH", right_index=True
    )

    if export_models:
        # export_models a True
        logger.debug("Saving models...START")
        stats_models_df.apply(
            lambda x: rnn_save(x.MODEL, path=_MODEL_PATH_ / "{}".format(x.MODEL_HASH)), axis=1
            )
        logger.debug("Saving models...DONE")

        param_df.reset_index().to_parquet(_MODEL_STATS_PATH_ / "df_params.parquet", engine="fastparquet")
        stats_df.reset_index().to_parquet(_MODEL_STATS_PATH_ / "df_stats.parquet", engine="fastparquet")
        stats_models_df.drop(columns=["MODEL"]).reset_index().to_parquet(_MODEL_STATS_PATH_ / "df_stats_models.parquet", engine="fastparquet")
        best_models_df.drop(columns=["MODEL"]).reset_index().to_parquet(_MODEL_STATS_PATH_ / "df_best_models.parquet", engine="fastparquet")

    return param_df, stats_df, stats_models_df, best_models_df

def train_models_on_df_refactor(
    ts: pd.DataFrame, 
    model_param: dict, 
    return_stats: bool=True, 
    save: bool=False
    ):
    from degiro.degiroUtils import rnn_train, rnn_save, rnn_get_stats_2
    from degiro.degiroUtils import from_dict_values_to_hash, build_dict_for_hashing
    import datetime

    param_hash = from_dict_values_to_hash(model_param)
    logger.debug("Param ID...{}".format(param_hash))

    chunks_len = int(len(ts.columns)/model_param["chunks_num"])
    logger.debug("Isin per chunk...{}".format(chunks_len))

    rnn_models = pd.DataFrame(index=ts.columns, columns=["MODEL", "SCALER_MIN", "SCALER_MAX", "TRAIN_LOSS", "TEST_LOSS", "VOL_IS", "VOL_OOS", "MODEL_HASH"], data=None)
    # TODO: rnn_models.index.name = "ISIN"
    for chunk_num in range(model_param["chunks_num"]):
        logger.debug("Start trainig chunk...{}".format(chunk_num+1))
        columns = ts.columns[(chunk_num*chunks_len):(chunk_num*chunks_len+chunks_len)]
        #logger.debug("Working on : ", list(columns))

        model_tmp = ts[columns].apply(
            rnn_train, 
            epochs=model_param["epochs"], 
            look_back=model_param["look_back"], 
            batch_size=model_param["batch_size"], 
            loss=model_param["loss"], 
            optimizer=model_param["optimizer"], 
            lstm_dim=model_param["lstm_dim"], 
            dense_dim=model_param["dense_dim"], 
            result_type="expand", 
            verbose=0
            )

        model_tmp_t = model_tmp.T
        model_tmp_t["MODEL_HASH"] = None
        rnn_models.loc[columns] = pd.DataFrame(data=model_tmp_t).rename(columns={
            0 : "MODEL",
            1 : "SCALER_MIN",
            2 : "SCALER_MAX",
            3 : "TRAIN_LOSS",
            4 : "TEST_LOSS",
            5 : "VOL_IS",
            6 : "VOL_OOS"
        })

        rnn_models.loc[columns, "TIMESTAMP"] = datetime.datetime.now().timestamp()

        # Building model specific index (hashing isin, timestamp and param)
        rnn_models.loc[columns, "MODEL_HASH"] = (rnn_models.loc[columns].reset_index().rename(columns={"index" : "ISIN"}).apply(
            lambda x : from_dict_values_to_hash(
                build_dict_for_hashing(model_param, {"isin" : x.ISIN, "time" : x.TIMESTAMP})
            ), axis=1)).values

        logger.debug("Done {} -> {}%".format(chunk_num + 1, (chunk_num + 1)/model_param["chunks_num"]*100))

    logger.debug(f"Getting stats...")
    rnn_models[["TRAIN_LOSS", "TEST_LOSS", "VOL_IS", "VOL_OOS"]] = rnn_models.apply(
        lambda x: rnn_get_stats_2(
            model=x.MODEL, 
            dataset=ts[x.name], 
            look_back=model_param["look_back"]
            ), 
            result_type="expand", axis=1
        )

    rnn_models["AVERAGE_PRICE"]   = ts.mean()
    rnn_models["LOOK_BACK"]       = model_param["look_back"]
    rnn_models["LAST_PRICE"]      = ts.iloc[-1]
    rnn_models["TRAIN_LOSS_PERC"] = rnn_models["TRAIN_LOSS"].div(rnn_models["AVERAGE_PRICE"])
    rnn_models["TEST_LOSS_PERC"]  = rnn_models["TEST_LOSS"].div(rnn_models["AVERAGE_PRICE"])
    rnn_models["VOL_IS_PERC"]     = rnn_models["VOL_IS"].div(rnn_models["AVERAGE_PRICE"])
    rnn_models["VOL_OOS_PERC"]    = rnn_models["VOL_OOS"].div(rnn_models["AVERAGE_PRICE"])
    rnn_models["PARAM_HASH"]      = param_hash
    
    if save:
        # Droppo colonna MODEL ed esporto in raw_file_path
        logger.debug(f"Writing session to parquet...")
        rnn_models.reset_index().drop(columns=["MODEL"]).to_parquet(_MODEL_STATS_PATH_ / "models_{}.parquet".format(param_hash), engine="fastparquet")

        logger.debug(f"Saving models...")
        rnn_models.apply(
            lambda x: rnn_save(x.MODEL, path=_MODEL_PATH_ / "{}".format(x.MODEL_HASH)), axis=1
            )
        logger.debug(f"Models saved...")

    if return_stats:
        return rnn_models, param_hash

In [11]:
pd.options.display.max_columns = 200

"""
model_path = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\models\\")
_MODEL_PATH_ = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\models\\")
models_stats_path = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\parquet\\")
_MODEL_STATS_PATH_ = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\parquet\\")
"""
raw_file_path = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\degiro\\degiro\\file\\raw\\")

degiro_conn = dg.DeGiro()
degiro_conn.login("Calsipher8", "Palazov13!")

{'data': {'id': 1858803,
  'intAccount': 111030591,
  'loggedInPersonId': 1861625,
  'clientRole': 'active',
  'effectiveClientRole': 'active',
  'isDebitMoneyEnabled': True,
  'contractType': 'PRIVATE',
  'username': 'Calsipher8',
  'displayName': 'OTTO FONTANESI',
  'email': 'otto.fonta@hotmail.it',
  'firstContact': {'firstName': 'OTTO',
   'lastName': 'FONTANESI',
   'displayName': 'OTTO FONTANESI',
   'nationality': 'IT',
   'gender': 'MALE',
   'dateOfBirth': '1990-02-11',
   'placeOfBirth': 'Sassuolo',
   'countryOfBirth': 'IT'},
  'address': {'streetAddress': 'Via Mezzacosta',
   'streetAddressNumber': '17',
   'zip': '40136',
   'city': 'Bologna',
   'country': 'IT'},
  'cellphoneNumber': 'REDACTED_PHONE',
  'locale': 'it_IT',
  'language': 'it',
  'culture': 'IT',
  'displayLanguage': 'it',
  'bankAccount': {'bankAccountId': 2796696,
   'bic': 'REDACTED_BIC',
   'iban': 'REDACTED_IBAN',
   'status': 'VERIFIED'},
  'flatexBankAccount': {'bic': 'REDACTED_BIC',
   'iban': 'REDAC

# GET TS AND ANAGR -> EXPORT dax_anagrafica.parquet / ts_dax.parquet

In [9]:
index_id=6
stock_country_id=906

In [ ]:
%%time
"""
S&P : 
    index_id=14
    stock_country_id=846
"""
downloader(
    conn_degiro=degiro_conn, 
    index_id=index_id, 
    stock_country_id=stock_country_id, 
    raw_file_path=raw_file_path,
    time_span="1Y"
    )

DEGIRO - download_stock_info - DEBUG - 0 : 41 - 3457
DEGIRO - download_stock_info - DEBUG - 1 : 41 - 886838
DEGIRO - download_stock_info - DEBUG - 2 : 41 - 4799
DEGIRO - download_stock_info - DEBUG - 3 : 41 - 3572
DEGIRO - download_stock_info - DEBUG - 4 : 41 - 3934
DEGIRO - download_stock_info - DEBUG - 5 : 41 - 3597
DEGIRO - download_stock_info - DEBUG - 6 : 41 - 3607
DEGIRO - download_stock_info - DEBUG - 7 : 41 - 883883
DEGIRO - download_stock_info - DEBUG - 8 : 41 - 3765
DEGIRO - download_stock_info - DEBUG - 9 : 41 - 8071694
DEGIRO - download_stock_info - DEBUG - 10 : 41 - 13697828
DEGIRO - download_stock_info - DEBUG - 11 : 41 - 3560
DEGIRO - download_stock_info - DEBUG - 12 : 41 - 3982
DEGIRO - download_stock_info - DEBUG - 13 : 41 - 3873
DEGIRO - download_stock_info - DEBUG - 14 : 41 - 3877
DEGIRO - download_stock_info - DEBUG - 15 : 41 - 7102
DEGIRO - download_stock_info - DEBUG - 16 : 41 - 144781
DEGIRO - download_stock_info - DEBUG - 17 : 41 - 3963
DEGIRO - download_stock_i

Wall time: 20.4 s


In [12]:
def updater(
    conn_degiro: degiroapi.DeGiro, 
    index_id : int, 
    stock_country_id : int, 
    raw_file_path: Path,
    time_span: str="1Y"
    ) -> None:
    ts = pd.read_parquet(raw_file_path / f"{index_id}_{stock_country_id}_ts.parquet", engine="fastparquet")

    logger.debug("Dim old > rows : {} - cols : {}".format(ts.shape[0], ts.shape[1]))
    logger.debug("Date old > last date : {} - last date : {}".format(ts.index[0], ts.index[-1]))
    
    # scrivo le tmp in raw_file_path / tmp
    downloader(
        conn_degiro=conn_degiro, 
        index_id=index_id, 
        stock_country_id=stock_country_id, 
        raw_file_path=raw_file_path / "tmp",
        time_span=time_span
        )

    ts_new = pd.read_parquet(raw_file_path / "tmp" / f"{index_id}_{stock_country_id}_ts.parquet", engine="fastparquet")
    ts_up_to_date = pd.concat([ts, ts_new])
    ts_up_to_date = ts_up_to_date[~ts_up_to_date.sort_index().index.duplicated(keep='first')]

    logger.debug("Dim old > rows : {} - cols : {}".format(ts.shape[0], ts.shape[1]))
    logger.debug("Date new > last obs : {} - first obs : {}".format(ts.index[0], ts.index[-1]))

    ts_up_to_date.to_parquet(raw_file_path / f"{index_id}_{stock_country_id}_ts.parquet", engine="fastparquet")
    logger.debug("Dim new > rows : {} - cols : {}".format(ts_up_to_date.shape[0], ts_up_to_date.shape[1]))
    logger.debug("Date new > last obs : {} - first obs : {}".format(ts_up_to_date.index[0], ts_up_to_date.index[-1]))

    return None
    

In [13]:
updater(
    degiro_conn, 
    index_id=index_id, 
    stock_country_id=stock_country_id, 
    raw_file_path=raw_file_path,
    time_span="1Y"
)

DEGIRO - updater - DEBUG - Dim old > rows : 377 - cols : 41
DEGIRO - updater - DEBUG - Date old > last date : 2021-05-14 00:00:00 - last date : 2022-08-04 00:00:00
DEGIRO - download_stock_info - DEBUG - 0 : 40 - 3457
DEGIRO - download_stock_info - DEBUG - 1 : 40 - 886838
DEGIRO - download_stock_info - DEBUG - 2 : 40 - 4799
DEGIRO - download_stock_info - DEBUG - 3 : 40 - 3572
DEGIRO - download_stock_info - DEBUG - 4 : 40 - 3934
DEGIRO - download_stock_info - DEBUG - 5 : 40 - 3597
DEGIRO - download_stock_info - DEBUG - 6 : 40 - 3607
DEGIRO - download_stock_info - DEBUG - 7 : 40 - 883883
DEGIRO - download_stock_info - DEBUG - 8 : 40 - 1893017
DEGIRO - download_stock_info - DEBUG - 9 : 40 - 3765
DEGIRO - download_stock_info - DEBUG - 10 : 40 - 8071694
DEGIRO - download_stock_info - DEBUG - 11 : 40 - 20979466
DEGIRO - download_stock_info - DEBUG - 12 : 40 - 3560
DEGIRO - download_stock_info - DEBUG - 13 : 40 - 3982
DEGIRO - download_stock_info - DEBUG - 14 : 40 - 3873
DEGIRO - download_stoc

In [14]:
ts_dax = pd.read_parquet(raw_file_path / f"{index_id}_{stock_country_id}_ts.parquet", engine="fastparquet")
ts_dax.index[0], ts_dax.index[-1]

(Timestamp('2021-05-14 00:00:00'), Timestamp('2023-01-30 00:00:00'))

In [15]:
ts_dax.shape

(420, 46)

In [16]:
ts_dax.head()

,DE000A1EWWW0,NL0000235190,DE0008404005,DE000BASF111,DE000BAY0017,DE0005190003,DE0005200000,DE000A1DAHH0,DE0005439004,DE0006062144,DE000A2E4K43,DE0005140008,DE0005810055,DE0005552004,DE0005557508,DE000A0HN5C6,DE000ENAG999,DE0005785802,DE0005785604,DE0006047004,DE000A161408,DE0006048432,DE0006231004,IE00BZ12WP82,DE0007100000,DE0006599905,DE000A0D9PT0,DE0008430026,DE000PAH0038,DE0006969603,NL0012169213,DE0007037129,DE0007164600,DE0007165631,DE0007236101,DE000ENER6Y0,DE000SHL1006,DE000SYM9999,DE0007664039,DE000A1ML7J1,DE000ZAL1111,DE000CBK1001,DE000DTR0CK8,DE0008402215,DE000PAG9113,DE0007030009
DATE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2021-05-14,180.48,104.60,198.34,49.035,60.280,75.78,93.66,69.68,65.04,38.99,27.14,9.1320,162.75,37.950,17.948,25.82,9.964,56.70,33.210,52.44,35.89,62.74,27.600,288.65,63.51,160.10,177.25,227.30,76.44,64.00,42.51,38.66,93.40,312.50,113.90,15.625,52.90,101.35,146.44,33.00,33.09,NaN,NaN,NaN,NaN,NaN
2021-05-15,181.86,107.14,205.15,49.675,57.895,82.26,93.50,68.60,64.02,39.16,25.58,9.3230,162.10,37.725,17.510,25.82,10.230,58.74,34.165,54.80,34.84,63.32,26.765,288.65,63.86,169.30,180.05,227.50,76.44,65.56,42.56,40.41,92.96,330.80,116.80,16.275,51.64,100.95,147.06,34.17,31.97,NaN,NaN,NaN,NaN,NaN
2021-05-16,173.36,105.02,193.72,48.450,61.680,78.21,90.64,67.36,61.80,38.43,25.22,9.1010,161.30,37.335,17.280,24.99,9.854,58.42,33.240,54.14,33.72,63.24,25.760,285.30,61.13,165.10,174.45,217.00,72.34,63.03,42.10,39.49,91.66,317.50,114.72,16.685,50.84,99.94,143.28,32.84,32.53,NaN,NaN,NaN,NaN,NaN
2021-05-17,173.68,104.48,191.26,47.150,58.720,77.82,88.92,65.82,60.82,37.50,25.64,8.9560,158.50,36.210,17.142,24.88,9.764,57.54,34.050,53.74,34.43,61.42,25.575,285.65,60.56,163.65,179.25,213.70,74.48,63.36,42.98,40.03,90.77,316.10,111.94,16.630,49.70,98.10,142.31,32.71,31.94,NaN,NaN,NaN,NaN,NaN
2021-05-18,179.98,105.84,198.22,49.110,62.900,77.33,95.75,70.34,64.62,39.44,29.67,9.2235,161.55,38.375,18.078,25.93,9.934,56.94,33.220,52.08,38.82,63.24,28.255,295.45,63.56,161.00,179.90,225.95,75.10,65.32,44.23,41.17,92.86,343.25,113.52,16.495,55.54,102.80,144.46,33.34,34.67,NaN,NaN,NaN,NaN,NaN


In [17]:
ts_dax.describe()

,DE000A1EWWW0,NL0000235190,DE0008404005,DE000BASF111,DE000BAY0017,DE0005190003,DE0005200000,DE000A1DAHH0,DE0005439004,DE0006062144,DE000A2E4K43,DE0005140008,DE0005810055,DE0005552004,DE0005557508,DE000A0HN5C6,DE000ENAG999,DE0005785802,DE0005785604,DE0006047004,DE000A161408,DE0006048432,DE0006231004,IE00BZ12WP82,DE0007100000,DE0006599905,DE000A0D9PT0,DE0008430026,DE000PAH0038,DE0006969603,NL0012169213,DE0007037129,DE0007164600,DE0007165631,DE0007236101,DE000ENER6Y0,DE000SHL1006,DE000SYM9999,DE0007664039,DE000A1ML7J1,DE000ZAL1111,DE000CBK1001,DE000DTR0CK8,DE0008402215,DE000PAG9113,DE0007030009
count,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,270.000000,420.000000,420.000000,420.000000,420.000000,270.000000,377.000000,270.000000,420.000000,420.000000,270.000000,420.000000,420.000000,270.000000,420.000000,420.000000,420.000000,420.000000,420.000000,270.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,202.819893,111.250119,203.560881,54.899554,53.867494,85.912964,100.333131,72.854952,78.123857,45.297536,77.530852,10.609464,157.832917,45.463101,18.323795,38.493130,10.289019,58.693537,32.507500,59.587631,60.364056,70.239405,32.525494,278.076111,68.771631,183.798929,197.951786,265.284583,72.274381,89.003352,44.720107,37.306821,108.488500,433.158452,133.004857,19.602113,54.121369,109.534512,157.555631,37.693423,52.681381,9.194303,29.435867,178.235667,105.732267,213.407500
std,64.258004,7.840358,14.881181,8.553851,5.817246,8.637339,9.548929,7.898672,20.019281,8.586703,37.881604,1.254454,11.064370,8.602245,1.732700,11.105305,1.198747,6.213649,7.014773,8.591350,22.820859,8.081693,5.104276,16.427314,8.336314,16.688141,19.291435,37.047894,14.010225,17.008567,2.157480,3.603779,12.657374,85.737535,15.529399,3.720125,5.470981,9.864600,28.980736,13.637487,24.245436,1.218573,2.051151,11.318900,9.272726,38.837484
min,93.950000,87.900000,159.620000,38.760000,44.190000,69.130000,83.360000,55.700000,44.835000,28.815000,25.220000,7.583500,134.400000,30.465000,15.380000,20.860000,7.420000,35.650000,20.040000,39.590000,25.740000,57.540000,21.925000,244.750000,50.620000,146.500000,153.200000,213.700000,50.060000,60.560000,39.610000,29.060000,81.870000,301.750000,95.450000,10.340000,42.010000,93.120000,114.880000,15.655000,19.415000,7.158000,22.695000,152.150000,82.520000,146.100000
25%,146.565000,105.840000,197.300000,47.808750,48.694375,78.607500,93.150000,67.540000,64.210000,37.702500,38.735000,9.740000,147.156250,38.230000,16.962000,27.665000,9.522000,56.075000,26.597500,52.485000,37.662500,63.170000,28.225000,264.425000,63.120000,171.650000,183.850000,237.150000,57.415000,70.970000,43.185000,34.265000,97.652500,366.000000,120.465000,17.186250,49.872500,102.187500,130.867500,24.260000,32.877500,8.046500,28.690000,171.500000,99.487500,185.937500
50%,187.980000,112.500000,203.537500,52.430000,53.230000,83.840000,98.520000,73.220000,68.925000,42.725000,78.540000,10.768000,159.750000,41.960000,17.938000,36.830000,10.316000,59.410000,32.245000,60.650000,57.350000,68.340000,32.930000,279.650000,68.845000,181.400000,196.125000,252.050000,76.560000,94.360000,44.377500,38.135000,108.640000,403.750000,137.860000,19.437500,54.160000,107.350000,148.560000,35.967500,42.770000,9.400000,30.032500,181.812500,106.700000,213.575000
75%,260.925000,115.647500,215.925000,63.777500,57.900000,92.590000,106.200000,78.635000,94.773750,54.450000,114.950000,11.420000,165.725000,54.020000,19.304000,52.380000,11.111000,61.320000,37.195000,65.460000,82.342500,77.040000,35.976250,292.962500,74.000000,193.862500,210.375000,299.425000,84.675000,104.200000,46.437500,40.380000,120.840000,522.250000,144.945000,22.812500,57.915000,117.450000,186.595000,50.520000,75.680000,10.025000,30.874375,186.162500,113.925000,247.687500
max,336.250000,129.280000,231.950000,68.750000,67.570000,108.

In [14]:
oo = ts_dax.isnull().apply(lambda col : col.any())

DE000A1EWWW0    False
NL0000235190    False
DE0008404005    False
DE000BASF111    False
DE000BAY0017    False
DE0005190003    False
DE0005200000    False
DE000A1DAHH0    False
DE0005439004    False
DE0006062144    False
DE000A2E4K43    False
DE0005140008    False
DE0005810055    False
DE0005552004    False
DE0005557508    False
DE000A0HN5C6    False
DE000ENAG999     True
DE0005785802    False
DE0005785604    False
DE0006047004    False
DE000A161408    False
DE0006048432    False
DE0006231004    False
IE00BZ12WP82    False
DE0007100000    False
DE0006599905    False
DE000A0D9PT0    False
DE0008430026    False
DE000PAH0038    False
DE0006969603    False
NL0012169213    False
DE0007037129    False
DE0007164600    False
DE0007165631    False
DE0007236101    False
DE000ENER6Y0    False
DE000SHL1006    False
DE000SYM9999    False
DE0007664039    False
DE000A1ML7J1    False
DE000ZAL1111    False
dtype: bool

# TRAIN / EXPORT / LOAD MODELS dax_models.parquet

In [19]:
raw_file_path = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\degiro\\degiro\\file\\raw\\")
index_id=6
stock_country_id=906

dax_anagrafica_test = pd.read_parquet(raw_file_path / f"{index_id}_{stock_country_id}_anagrafica.parquet", engine="fastparquet")
ts_dax = pd.read_parquet(raw_file_path / f"{index_id}_{stock_country_id}_ts.parquet", engine="fastparquet")

In [20]:
# MANCA '2021-10-09'
ts_dax.loc['2021-10-10']

,DE000A1EWWW0,NL0000235190,DE0008404005,DE000BASF111,DE000BAY0017,DE0005190003,DE0005200000,DE000A1DAHH0,DE0005439004,DE0006062144,DE000A2E4K43,DE0005140008,DE0005810055,DE0005552004,DE0005557508,DE000A0HN5C6,DE000ENAG999,DE0005785802,DE0005785604,DE0006047004,DE000A161408,DE0006048432,DE0006231004,IE00BZ12WP82,DE0007100000,DE0006599905,DE000A0D9PT0,DE0008430026,DE000PAH0038,DE0006969603,NL0012169213,DE0007037129,DE0007164600,DE0007165631,DE0007236101,DE000ENER6Y0,DE000SHL1006,DE000SYM9999,DE0007664039,DE000A1ML7J1,DE000ZAL1111,DE000CBK1001,DE000DTR0CK8,DE0008402215,DE000PAG9113,DE0007030009
DATE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2021-10-10,171.88,106.52,194.8,48.64,63.38,77.84,92.82,69.06,65.78,39.61,29.55,9.492,159.15,37.48,18.08,26.02,9.956,55.82,31.79,52.02,33.81,61.4,27.845,293.95,63.28,167.7,181.9,222.8,70.86,62.0,42.115,42.47,90.23,337.7,112.46,17.08,54.36,101.3,145.32,33.8,36.02,NaN,NaN,NaN,NaN,NaN


In [21]:
ts_dax[['DE000A1EWWW0', 'DE0008404005', 'DE000BASF111', 'DE000BAY0017']].iloc[-5:]

,DE000A1EWWW0,DE0008404005,DE000BASF111,DE000BAY0017
DATE,,,,
2023-01-23,121.86,164.94,41.325,47.845
2023-01-24,122.58,166.14,41.920,48.625
2023-01-25,126.20,167.20,42.300,48.715
2023-01-29,118.88,161.42,39.625,47.520
2023-01-30,124.02,159.62,38.760,47.625


## Load params

In [45]:
%%time
"""
Tests:
    1)
        4 x -> Wall time: 9min 32s

    2)
        6 x -> --- 00:10:50 ---> Total time (min) spent on 6 iterations
            Saving models...START
            Wall time: 21min 33s <- 4 models to save

    3)
        4 isin; --> export_models=False <-- ;epochs=3000;look_back=10;lstm_dim=40;dense_dim=10
            test 1 : --- 00:02:54 ---> Total time (min) spent on 1 iterations
            test 2 : --- 00:02:28 ---> Total time (min) spent on 1 iterations

    4)
        10 isin;export_models=False; -->  epochs=30000  <-- ; -->  look_back=100 <-- ; --> lstm_dim=400 <-- ; --> dense_dim=100 <--
            test 1 : 
            test 2 : 

"""
n_obs   = 252
to_test = ['DE000A1EWWW0', 'DE0008404005', 'DE000BASF111', 'DE000BAY0017']
# to_test = ts_dax.columns

model_param = {
    "chunks_num" : 2
}

model_param.update({
    "look_back" : 10,
    "epochs" : 5,
    "loss" : "mean_squared_error",
    "optimizer" : "adam",
    "lstm_dim" : 4,
    "dense_dim" : 1,
    "batch_size" : 1
})

grid_param = {
    "look_back" : [5],
    "epochs" : [
        10000, 5000
        ],
    "loss" : ["mean_squared_error"],
    "optimizer" : ["adam"],
    "lstm_dim" : [8],
    "dense_dim" : [32],
    "batch_size" : [150],
}

Wall time: 0 ns


In [49]:
"""
Ritorna (tra parentesi equivalente output da fct train_on_grid):
    df1 (grid_output_param) : 
        df index PARAM_HASH value parametrizzazione rete
    df2 (grid_output_stats) : 
        df index [PARAM_HASH, STAT] (STAT -> headers metodo pd.DataFrame.describe()) value pd.DataFrame.describe() output
    df3 (models_df) : 
        df no index (ISIN, PARAM_HASH da settare) con [ISIN, MODEL, MODEL_HASH, PARAM_HASH, statistiche performance model]
    df4 : 
        df no index (ISIN da settare) con miglior setting parametri e stat (merge df1 e df3)
"""
df1, df2, df3, df4 = train_on_grid_refactor(
    global_param=model_param, 
    grid_param=grid_param, 
    ts=ts_dax[to_test].iloc[-n_obs:], 
    export_models=True
    )

df1.shape, df2.shape, df3.shape, df4.shape

DEGIRO - train_on_grid_refactor - DEBUG - Total iterations...2
DEGIRO - train_on_grid_refactor - DEBUG - [{'batch_size': 150, 'dense_dim': 32, 'epochs': 10000, 'look_back': 5, 'loss': 'mean_squared_error', 'lstm_dim': 8, 'optimizer': 'adam'}, {'batch_size': 150, 'dense_dim': 32, 'epochs': 5000, 'look_back': 5, 'loss': 'mean_squared_error', 'lstm_dim': 8, 'optimizer': 'adam'}]
DEGIRO - train_on_grid_refactor - DEBUG - {'batch_size': 150, 'dense_dim': 32, 'epochs': 10000, 'look_back': 5, 'loss': 'mean_squared_error', 'lstm_dim': 8, 'optimizer': 'adam'}
DEGIRO - train_models_on_df_refactor - DEBUG - Param ID...a0b30aff28c5468c2f91e2a328f255ff
DEGIRO - train_models_on_df_refactor - DEBUG - Isin per chunk...2
DEGIRO - train_models_on_df_refactor - DEBUG - Start trainig chunk...1
DEGIRO - train_models_on_df_refactor - DEBUG - Done 1 -> 50.0%
DEGIRO - train_models_on_df_refactor - DEBUG - Start trainig chunk...2
DEGIRO - train_models_on_df_refactor - DEBUG - Done 2 -> 100.0%
DEGIRO - train_mo

--- 00:03:46 ---> Execution time (min) : last iteration
--- 00:03:46 ---> Average execution time (min) for 1 iterations
--- 00:03:46 ---> Remaining estimated time (min) for 1 iterations
--- 00:03:46 ---> Total time (min) spent on 1 iterations


DEGIRO - train_models_on_df_refactor - DEBUG - Done 1 -> 50.0%
DEGIRO - train_models_on_df_refactor - DEBUG - Start trainig chunk...2
DEGIRO - train_models_on_df_refactor - DEBUG - Done 2 -> 100.0%
DEGIRO - train_models_on_df_refactor - DEBUG - Getting stats...
DEGIRO - train_on_grid_refactor - DEBUG - Done 2 -> 100.0%
DEGIRO - train_on_grid_refactor - DEBUG - Saving models...START


--- 00:01:59 ---> Execution time (min) : last iteration
--- 00:02:53 ---> Average execution time (min) for 2 iterations
--- 00:00:00 ---> Remaining estimated time (min) for 0 iterations
--- 00:05:46 ---> Total time (min) spent on 2 iterations


DEGIRO - train_on_grid_refactor - DEBUG - Saving models...DONE


((2, 7), (16, 4), (8, 18), (4, 25))

In [50]:
ts_dax[to_test].DE000A1EWWW0.values[0]

180.48

In [51]:
import struct

b = bytes()
b = struct.pack('f', ts_dax[to_test].DE000A1EWWW0.values[0])
struct.unpack("f", b)

(180.47999572753906,)

In [52]:
struct.unpack("f", struct.pack("f", ts_dax[to_test].DE000A1EWWW0.values[0]))

(180.47999572753906,)

In [53]:
l = list()
ll = list()
l.extend(struct.pack('f', val) for val in ts_dax[to_test].DE000A1EWWW0.values)
ll.extend(round(struct.unpack("f", val)[0], 2) for val in l)

In [54]:
l[:10], ll[:10], ts_dax[to_test].DE000A1EWWW0.values[:10]

([b'\xe1z4C',
  b')\xdc5C',
  b')\\-C',
  b'\x14\xae-C',
  b'\xe1\xfa3C',
  b'\x9aYRC',
  b'3\xb3KC',
  b'f&RC',
  b'f&gC',
  b'\x00@rC'],
 [180.48,
  181.86,
  173.36,
  173.68,
  179.98,
  210.35,
  203.7,
  210.15,
  231.15,
  242.25],
 array([180.48, 181.86, 173.36, 173.68, 179.98, 210.35, 203.7 , 210.15,
        231.15, 242.25]))

In [55]:
for a in bytearray(ts_dax[to_test].DE000A1EWWW0.values[:10]):
    logger.debug(a)

DEGIRO - <module> - DEBUG - 143
DEGIRO - <module> - DEBUG - 194
DEGIRO - <module> - DEBUG - 245
DEGIRO - <module> - DEBUG - 40
DEGIRO - <module> - DEBUG - 92
DEGIRO - <module> - DEBUG - 143
DEGIRO - <module> - DEBUG - 102
DEGIRO - <module> - DEBUG - 64
DEGIRO - <module> - DEBUG - 236
DEGIRO - <module> - DEBUG - 81
DEGIRO - <module> - DEBUG - 184
DEGIRO - <module> - DEBUG - 30
DEGIRO - <module> - DEBUG - 133
DEGIRO - <module> - DEBUG - 187
DEGIRO - <module> - DEBUG - 102
DEGIRO - <module> - DEBUG - 64
DEGIRO - <module> - DEBUG - 236
DEGIRO - <module> - DEBUG - 81
DEGIRO - <module> - DEBUG - 184
DEGIRO - <module> - DEBUG - 30
DEGIRO - <module> - DEBUG - 133
DEGIRO - <module> - DEBUG - 171
DEGIRO - <module> - DEBUG - 101
DEGIRO - <module> - DEBUG - 64
DEGIRO - <module> - DEBUG - 246
DEGIRO - <module> - DEBUG - 40
DEGIRO - <module> - DEBUG - 92
DEGIRO - <module> - DEBUG - 143
DEGIRO - <module> - DEBUG - 194
DEGIRO - <module> - DEBUG - 181
DEGIRO - <module> - DEBUG - 101
DEGIRO - <module> -

## Inizio a capirci qualcosa, 

### pare che:

    2 funzioni, train_on_grid e train_on_grid_refactor, fanno entrambe (le strambe) fondamentalmente la stessa 
    
    cosa, runnare modelli sul dataframe in input, scrivono su disco i modelli (checkpoint? Restart?) e 
    
    restituiscono outputs.

    inputs:
        train_on_grid(
            global_param, 
            grid_param, 
            ts, 
            return_param_hash=True,         -> input di train_models_on_df, per farsi tornare lista contenete la variabile rnn_models["PARAM_HASH"] = param_hash
            return_stats=True               -> input di train_models_on_df, returns rnn_models[["TRAIN_LOSS", "TEST_LOSS", "VOL_IS", "VOL_OOS"]].describe()
            )
        
        train_on_grid_refactor(
            global_param: dict, 
            grid_param: dict, 
            ts: pd.DataFrame, 

            return_stats: bool=True,        -> inutilissimo orrido modo per farsi tornare come output it_stats_models, param_hash da train_models_on_df_refactor, conteneti le statistiche del chunk e il vettore con le param_hash. Tale vettore Ã¨ la colonna PARAM_HASH del dataframe it_stats_models. Dio cane
            
            save: bool=False,               -> input di train_models_on_df_refactor dio cane
                nda: 
                    # Giro con save e return_stats defaultati a False e True -> traduzione :  train_models_on_df_refactor non scrive nulla : ovvero, save salva i modelli del chuck in pasto a train_models_on_df_refactor. Le seguenti due righe non sono eseguite.
                        raw_file_path / "models_{}.parquet".format(param_hash)
                        rnn_save(x.MODEL, path=_MODEL_PATH_ / "{}".format(x.MODEL_HASH)

            export_models: bool=False       -> rnn_save(x.MODEL, path=_MODEL_PATH_ / "{}".format(x.MODEL_HASH)), axis=1
                nda:
                    satanicamente defaultato a False, ma passato ovunque a True.
                    esegue il metodo rnn_save(x.MODEL, path=_MODEL_PATH_ / "{}".format(x.MODEL_HASH) all interno di un apply
            )

    Outputs:
        train_on_grid
            param_df: pd.DataFrame 
                index : [PARAM_HASH]
                columns : [batch_size, dense_dim, epochs, look_back, loss, lstm_dim, optimizer,chunks_num]

            stats_df: pd.DataFrame
                index : [PARAM_HASH, STAT]
                columns : [TRAIN_LOSS	TEST_LOSS	VOL_IS	VOL_OOS]
        
        train_on_grid_refactor
            param_df: pd.DataFrame 
                index : [PARAM_HASH]
                columns : [batch_size, dense_dim, epochs, look_back, loss, lstm_dim, optimizer,chunks_num]

            stats_df: pd.DataFrame
                index : [PARAM_HASH, STAT]
                columns : [TRAIN_LOSS	TEST_LOSS	VOL_IS	VOL_OOS]

            stats_models_df: pd.DataFrame
                index: []
                columns: [ISIN	MODEL	SCALER_MIN	SCALER_MAX	TRAIN_LOSS	TEST_LOSS	VOL_IS	VOL_OOS	MODEL_HASH	TIMESTAMP	AVERAGE_PRICE	LOOK_BACK	LAST_PRICE	TRAIN_LOSS_PERC	TEST_LOSS_PERC	VOL_IS_PERC	VOL_OOS_PERC	PARAM_HASH]
            
            best_models_df: pd.DataFrame
                query eseguita
                    stats_models_df.loc[
                        stats_models_df.groupby("ISIN")[_BEST_CHOICE_CRITERIA_].idxmin()
                        ].sort_values("PARAM_HASH").merge(
                            param_df, 
                            left_on="PARAM_HASH", 
                            right_index=True
                            )
                    _BEST_CHOICE_CRITERIA_ : variabile brutalmente hardcodata
                index: []
                columns: ['ISIN', 'MODEL', 'SCALER_MIN', 'SCALER_MAX', 'TRAIN_LOSS', 'TEST_LOSS','VOL_IS', 'VOL_OOS', 'MODEL_HASH', 'TIMESTAMP', 'AVERAGE_PRICE', 'LOOK_BACK', 'LAST_PRICE', 'TRAIN_LOSS_PERC', 'TEST_LOSS_PERC', 'VOL_IS_PERC', 'VOL_OOS_PERC', 'PARAM_HASH', 'batch_size', 'dense_dim', 'epochs', 'look_back', 'loss', 'lstm_dim', 'optimizer', 'chunks_num']

    Commenti a caldo:
        input:
            train_on_grid
                perchÃ¨ 2 param?
                perchÃ¨ 2 varibili (return_param_hash), return_stats defaultizzate a True? Rimuovere?

        output:
            train_on_grid -> param_df = train_on_grid_refactor -> param_df
            train_on_grid -> stats_df = train_on_grid_refactor -> stats_df
            train_on_grid_refactor -> stats_models_df 
                nda: 
                 pare vero che: 
                    df_3_ref.set_index(["PARAM_HASH", "ISIN"]).drop(columns=["MODEL"])
                    ==
                    output di resume_models_df(param_df, _MODEL_STATS_PATH_)
                sia Vero
            train_on_grid_refactor -> best_models_df 
                nda:
                    merge tra param_df e stats_models_df su PARAM_HASH
        
        file:
            train_on_grid 


        Dove cazzo sono i modelli
            df_3_ref e df_4_ref, colonna MODEL
            scritti in _MODEL_PATH_ : "C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\processed\\parquet\\"

train_models_on_df(
    ts : pd.DataFrame, 
    model_param: dict, 
    return_param_hash=False, 
    return_stats=False
    )

train_models_on_df_refactor(
    ts: pd.DataFrame, 
    model_param: dict, 
    return_stats: bool=True, 
    save: bool=False
    )

### Stessi hyperparam per tutto il set di isin

    PRO : 
        stesso look_back cross isin -> serie storiche pari

    CONTRO:
        un grado di liberta in meno - meno accurato

In [6]:
raw_file_path = Path("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\degiro\\degiro\\file\\raw\\")
index_id=6
stock_country_id=906

ts_dax = pd.read_parquet(raw_file_path / f"{index_id}_{stock_country_id}_ts.parquet", engine="fastparquet")

n_obs   = 252
to_test = ['DE000A1EWWW0', 'DE0008404005', 'DE000BASF111', 'DE000BAY0017']
# to_test = ts_dax.columns

model_param = {
    "chunks_num" : 2
}

model_param.update({
    "look_back" : 10,
    "epochs" : 5,
    "loss" : "mean_squared_error",
    "optimizer" : "adam",
    "lstm_dim" : 4,
    "dense_dim" : 1,
    "batch_size" : 1
})

grid_param = {
    "look_back" : [5],
    "epochs" : [
        10000
        ],
    "loss" : ["mean_squared_error"],
    "optimizer" : ["adam"],
    "lstm_dim" : [8],
    "dense_dim" : [32],
    "batch_size" : [150],
}

_BEST_ = "TEST_LOSS"

In [58]:
df1.reset_index().to_parquet(_MODEL_STATS_PATH_ / "df_1_ref.parquet", engine="fastparquet")
df2.reset_index().to_parquet(_MODEL_STATS_PATH_ / "df_2_ref.parquet", engine="fastparquet")
df3.drop(columns=["MODEL"]).to_parquet(_MODEL_STATS_PATH_ / "df_3_ref.parquet", engine="fastparquet")
df4.drop(columns=["MODEL"]).to_parquet(_MODEL_STATS_PATH_ / "df_4_ref.parquet", engine="fastparquet")

In [4]:
df_1_ref = pd.read_parquet(_MODEL_STATS_PATH_ / "df_1_ref.parquet", engine="fastparquet")
df_1_ref.set_index("PARAM_HASH", inplace=True)
df_2_ref = pd.read_parquet(_MODEL_STATS_PATH_ / "df_2_ref.parquet", engine="fastparquet")

In [8]:
ordered_best = df_2_ref[df_2_ref.STAT=="mean"][[_BEST_, "PARAM_HASH"]].sort_values(_BEST_, ascending=True).PARAM_HASH
df_1_ref.reindex(ordered_best)

,batch_size,dense_dim,epochs,look_back,loss,lstm_dim,optimizer
PARAM_HASH,,,,,,,
8985324fa42104418279f22abd280ded,150,32,5000,5,mean_squared_error,8,adam
a0b30aff28c5468c2f91e2a328f255ff,150,32,10000,5,mean_squared_error,8,adam


In [10]:
df_2_ref[df_2_ref.STAT=="mean"][[_BEST_, "PARAM_HASH"]].sort_values(_BEST_, ascending=True)

,TEST_LOSS,PARAM_HASH
9,2.500215,8985324fa42104418279f22abd280ded
1,2.935839,a0b30aff28c5468c2f91e2a328f255ff


## samples.parquet

## ground_truth.parquet

## ground_prediction.parquet

In [61]:
%load_ext autoreload
%autoreload 2
%aimport src_config

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from degiro.degiroUtils import rnn_load, rnn_predict_dataset
import copy

look_back = 5
ts_to_test = copy.deepcopy(ts_dax[to_test])

In [8]:
ts_to_test.iloc[-n_obs:]

,DE000A1EWWW0,DE0008404005,DE000BASF111,DE000BAY0017
DATE,,,,
2021-08-05,171.00,179.18,43.935,58.680
2021-08-07,171.34,178.28,43.440,56.730
2021-08-10,167.80,177.32,43.445,57.090
2021-08-11,166.34,173.38,42.520,57.080
2021-08-18,175.08,173.82,43.615,57.230
...,...,...,...,...
2023-01-23,121.86,164.94,41.325,47.845
2023-01-24,122.58,166.14,41.920,48.625
2023-01-25,126.20,167.20,42.300,48.715


In [9]:
df_4_ref = pd.read_parquet(path=_MODEL_STATS_PATH_ / "df_4_ref.parquet", engine="fastparquet").set_index("ISIN")
df_4_ref["MODEL"] = df_4_ref.apply(lambda x: rnn_load(path=_MODEL_PATH_ / x.MODEL_HASH), axis=1)
df_4_ref.head()

,SCALER_MIN,SCALER_MAX,TRAIN_LOSS,TEST_LOSS,VOL_IS,VOL_OOS,MODEL_HASH,TIMESTAMP,AVERAGE_PRICE,LOOK_BACK,...,VOL_OOS_PERC,PARAM_HASH,batch_size,dense_dim,epochs,look_back,loss,lstm_dim,optimizer,MODEL
ISIN,,,,,,,,,,,,,,,,,,,,,
DE0008404005,159.62,231.95,4.220403,3.151327,4.219984,2.360025,ad79af16fa4a35bc9f40e64720a5b921,1.685540e+09,203.482004,5,...,0.011598,8985324fa42104418279f22abd280ded,150,32,5000,5,mean_squared_error,8,adam,<keras.engine.sequential.Sequential object at ...
DE000A1EWWW0,93.95,316.00,13.466767,5.102551,13.466761,4.555416,55310890d0ce5bbc0f7bd0dd6e08c123,1.685540e+09,173.372004,5,...,0.026275,8985324fa42104418279f22abd280ded,150,32,5000,5,mean_squared_error,8,adam,<keras.engine.sequential.Sequential object at ...
DE000BASF111,38.76,68.64,2.257235,0.894146,2.203925,0.892919,de3d8542b533fddafd96e191b4d68491,1.685540e+09,51.710764,5,...,0.017268,8985324fa42104418279f22abd280ded,150,32,5000,5,mean_squared_error,8,adam,<keras.engine.sequential.Sequential object at ...
DE000BAY0017,44.57,67.42,1.382214,0.852835,1.381831,0.852752,bc27d82eb8f394240da95cc302dd95cf,1.685540e+09,54.552589,5,...,0.015632,8985324fa42104418279f22abd280ded,150,32,5000,5,mean_squared_error,8,adam,<keras.engine.sequential.Sequential object at ...


In [10]:
df_3_ref = pd.read_parquet(path=_MODEL_STATS_PATH_ / "df_3_ref.parquet", engine="fastparquet").set_index("ISIN")
df_3_ref["MODEL"] = df_3_ref.apply(lambda x: rnn_load(path=_MODEL_PATH_ / x.MODEL_HASH), axis=1)
df_3_ref.head()

,SCALER_MIN,SCALER_MAX,TRAIN_LOSS,TEST_LOSS,VOL_IS,VOL_OOS,MODEL_HASH,TIMESTAMP,AVERAGE_PRICE,LOOK_BACK,LAST_PRICE,TRAIN_LOSS_PERC,TEST_LOSS_PERC,VOL_IS_PERC,VOL_OOS_PERC,PARAM_HASH,MODEL
ISIN,,,,,,,,,,,,,,,,,
DE000A1EWWW0,93.95,316.00,13.312616,6.380830,13.273747,4.959042,677e419151ac0856b54850e3ddf79479,1.685540e+09,173.372004,5,124.020,0.076786,0.036804,0.076562,0.028603,a0b30aff28c5468c2f91e2a328f255ff,<keras.engine.sequential.Sequential object at ...
DE0008404005,159.62,231.95,4.165190,3.501867,4.155565,2.503167,406adba7eb5bca863f8be974244041bc,1.685540e+09,203.482004,5,159.620,0.020470,0.017210,0.020422,0.012302,a0b30aff28c5468c2f91e2a328f255ff,<keras.engine.sequential.Sequential object at ...
DE000BASF111,38.76,68.64,2.217020,0.958115,2.197068,0.943279,d74e3610b0576abd60853c774927588b,1.685540e+09,51.710764,5,38.760,0.042873,0.018528,0.042488,0.018241,a0b30aff28c5468c2f91e2a328f255ff,<keras.engine.sequential.Sequential object at ...
DE000BAY0017,44.57,67.42,1.392718,0.902543,1.373615,0.890172,19587c7f8a1270b237e3cb0659b7d6bc,1.685540e+09,54.552589,5,47.625,0.025530,0.016544,0.025180,0.016318,a0b30aff28c5468c2f91e2a328f255ff,<keras.engine.sequential.Sequential object at ...
DE000A1EWWW0,93.95,316.00,13.466767,5.102551,13.466761,4.555416,55310890d0ce5bbc0f7bd0dd6e08c123,1.685540e+09,173.372004,5,124.020,0.077676,0.029431,0.077676,0.026275,8985324fa42104418279f22abd280ded,<keras.engine.sequential.Sequential object at ...


In [11]:
ts_dax.loc['2021-10-09']
ts_to_test.iloc[-n_obs:].shape

(252, 4)

In [12]:
%%time
df_3_ref[["PREDICTED_Y", "TEST_Y"]] = df_3_ref.apply(lambda x: rnn_predict_dataset(
    model=x.MODEL, 
    scaler_min=x.SCALER_MIN, 
    scaler_max=x.SCALER_MAX, 
    dataset=ts_to_test[x.name][int(ts_to_test.iloc[-n_obs:].shape[0]*0.67):], 
    in_sample_perc=0, 
    look_back=look_back
    ), 
    result_type="expand", axis=1)

Wall time: 2.88 s


In [15]:
last_date

Timestamp('2023-01-25 00:00:00')

In [14]:
#df_4_ref
#ts_to_test.iloc[-n_obs:]

sample_col = [f"SAMPLE_{sample}" for sample in range(ts_to_test.iloc[-n_obs:].shape[0]-look_back)]
date_col = [(ts_to_test.iloc[-n_obs:].index[positional:(positional+look_back)]) for positional, date in enumerate(ts_to_test.iloc[-n_obs:].index[:-look_back])]

columns = []
for c, sample in enumerate(sample_col):
    for obs_date in date_col[c]:
        columns.append((sample, obs_date))

ss = pd.DataFrame(index=ts_to_test.iloc[-n_obs:].columns, columns=pd.MultiIndex.from_tuples(columns))
last_date = ss.columns.get_level_values(level=1)[-2]

In [80]:
date_col = [(ts_to_test.iloc[-n_obs:].index[positional:(positional+look_back)]) for positional, date in enumerate(ts_to_test.iloc[-n_obs:].index[:-look_back])]
date_col

[DatetimeIndex(['2021-08-05', '2021-08-07', '2021-08-10', '2021-08-11',
                '2021-08-18'],
               dtype='datetime64[ns]', name='DATE', freq=None),
 DatetimeIndex(['2021-08-07', '2021-08-10', '2021-08-11', '2021-08-18',
                '2021-08-20'],
               dtype='datetime64[ns]', name='DATE', freq=None),
 DatetimeIndex(['2021-08-10', '2021-08-11', '2021-08-18', '2021-08-20',
                '2021-08-24'],
               dtype='datetime64[ns]', name='DATE', freq=None),
 DatetimeIndex(['2021-08-11', '2021-08-18', '2021-08-20', '2021-08-24',
                '2021-08-25'],
               dtype='datetime64[ns]', name='DATE', freq=None),
 DatetimeIndex(['2021-08-18', '2021-08-20', '2021-08-24', '2021-08-25',
                '2021-08-26'],
               dtype='datetime64[ns]', name='DATE', freq=None),
 DatetimeIndex(['2021-08-20', '2021-08-24', '2021-08-25', '2021-08-26',
                '2021-08-28'],
               dtype='datetime64[ns]', name='DATE', freq=None)

In [77]:
# ts_to_test.iloc[-n_obs:].index[0:(0+look_back)]
# ts_to_test.iloc[-n_obs:].loc[['2021-10-01', '2021-10-03', '2021-10-07', '2021-10-08',
#               '2021-10-09']].T
ss.head()

SAMPLE_0                                              \
             2021-08-05 2021-08-07 2021-08-10 2021-08-11 2021-08-18   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_1                                              \
             2021-08-07 2021-08-10 2021-08-11 2021-08-18 2021-08-20   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_2                                              \
             2021-08-10 2021-08-11 2021-08-18 2021-08-20 2021-08-24   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_3                                              \
             2021-08-11 2021-08-18 2021-08-20 2021-08-24 2021-08-25   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_4                                              \
             2021-08-18 2021-08-20 2021-08-24 2021-08-25 2021-08-26   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_5                                              \
             2021-08-20 2021-08-24 2021-08-25 2021-08-26 2021-08-28   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_6                                              \
             2021-08-24 2021-08-25 2021-08-26 2021-08-28 2021-08-31   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_7                                              \
             2021-08-25 2021-08-26 2021-08-28 2021-08-31 2021-09-07   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_8                                              \
             2021-08-26 2021-08-28 2021-08-31 2021-09-07 2021-09-10   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

               SAMPLE_9                                              \
             2021-08-28 2021-08-31 2021-09-07 2021-09-10 2021-09-11   
DE000A1EWWW0        NaN       

In [72]:
for i, obs_date in enumerate(ts_to_test.iloc[-n_obs:].index[:len(date_col)]):
    print(obs_date, last_date)
    if obs_date >= last_date:
        break
    else:
        print(obs_date)
        date_window = ts_to_test.iloc[-n_obs:].index[i:(i+look_back)]
        cols = ss.columns[ss.columns.get_level_values(level=0)==f"SAMPLE_{i}"]
        print(ts_to_test.iloc[-n_obs:].loc[date_window].T.shape)
        print(ts_to_test.iloc[-n_obs:].loc[date_window].T)
        print(cols)
        ss[cols] = ts_to_test.iloc[-n_obs:].loc[date_window].T.astype(np.float)

2021-08-05 00:00:00 2023-01-25 00:00:00
2021-08-05 00:00:00
(4, 5)
DATE          2021-08-05  2021-08-07  2021-08-10  2021-08-11  2021-08-18
DE000A1EWWW0     171.000      171.34     167.800      166.34     175.080
DE0008404005     179.180      178.28     177.320      173.38     173.820
DE000BASF111      43.935       43.44      43.445       42.52      43.615
DE000BAY0017      58.680       56.73      57.090       57.08      57.230
MultiIndex([('SAMPLE_0', '2021-08-05'),
            ('SAMPLE_0', '2021-08-07'),
            ('SAMPLE_0', '2021-08-10'),
            ('SAMPLE_0', '2021-08-11'),
            ('SAMPLE_0', '2021-08-18')],
           )
2021-08-07 00:00:00 2023-01-25 00:00:00
2021-08-07 00:00:00
(4, 5)
DATE          2021-08-07  2021-08-10  2021-08-11  2021-08-18  2021-08-20
DE000A1EWWW0      171.34     167.800      166.34     175.080      175.92
DE0008404005      178.28     177.320      173.38     173.820      176.02
DE000BASF111       43.44      43.445       42.52      43.615       4

ValueError: Columns must be same length as key

In [110]:
def build_predicted_df(models_df: pd.DataFrame, ts: pd.DataFrame, look_back: int):
    """
    TODO:
        Fct da wrappare passando come input :
            fare distinct valori look_back
            splittare models_df e ts (entrambi hanno index isin) per livello di look_back relativo a tali prodotti
            invocare build_predicted_df con look_back e relativo ritaglio di models_df e ts
    """
    from degiro.degiroUtils import sample_field_parser, get_prediction

    # Genero ss : columns -> columns di ss; ts.columns -> index di ss
    sample_col = [f"SAMPLE_{sample}" for sample in range(ts.shape[0]-look_back)]
    date_col = [(ts.index[positional:(positional+look_back)]) for positional, date in enumerate(ts.index[:-look_back])]

    columns = []
    for c, sample in enumerate(sample_col):
        for obs_date in date_col[c]:
            columns.append((sample, obs_date))

    ss = pd.DataFrame(index=ts.columns, columns=pd.MultiIndex.from_tuples(columns))
    last_date = ss.columns.get_level_values(level=1)[-2]
    for i, obs_date in enumerate(ts.index[:len(date_col)]):
        if obs_date >= last_date:
            logger.debug(obs_date, last_date)
            break
        try:
            date_window = ts.index[i:(i+look_back)]
            cols = ss.columns[ss.columns.get_level_values(level=0)==f"SAMPLE_{i}"]
            ss[cols] = ts.loc[date_window].T.astype(np.float)
            
        except:
            #logger.debug(i)
            #logger.debug(date_window)
            logger.debug(cols)

    ss_T = ss.T
    ss_T.index.set_names(["SAMPLE", "DATE"], inplace=True)
    ss_T["Y"] = "GROUND_TRUE"
    ss_T.reset_index("SAMPLE", inplace=True)
    ss_T.rename(columns={"SAMPLE" : "SAMPLE_OLD"}, inplace=True)
    ss_T["SAMPLE"] = ss_T.SAMPLE_OLD.apply(sample_field_parser, look_back=look_back)

    ground_truth = ss_T.reset_index().groupby("SAMPLE_OLD").head(1).iloc[look_back:-1].set_index(["Y", "SAMPLE", "DATE"]).T
    pp = ss.T.reset_index("DATE").iloc[:-look_back*(look_back+1)].set_index("DATE", append=True)
    pp["Y"] = "SAMPLES"
    pp = pp.reset_index().set_index(["Y", "SAMPLE", "DATE"])
    ss = pp.T
    ss = pd.merge(ss, ground_truth, how="inner", left_index=True, right_index=True)

    ground_prediction = pd.DataFrame(index=ss[("GROUND_TRUE")].index, columns=ss[("GROUND_TRUE")].columns, data=None)
    tot_stocks=len(ground_prediction.index)
    for c, isin in enumerate(ground_prediction.index):
        for sample, data in zip(ground_prediction.columns.get_level_values(0), ground_prediction.columns.get_level_values(1)):
            ground_prediction.loc[isin, (sample, data)] = get_prediction(isin, sample, models_df, ss)
        logger.debug(round(c/tot_stocks, 4))

    samples = ss[("SAMPLES")]

    return samples, ground_truth, ground_prediction

In [30]:
def build_predicted_df_OLD(models_df: pd.DataFrame, ts: pd.DataFrame, look_back: int):
    """OLD
    TODO:
        Fct da wrappare passando come input :
            fare distinct valori look_back
            splittare models_df e ts (entrambi hanno index isin) per livello di look_back relativo a tali prodotti
            invocare build_predicted_df con look_back e relativo ritaglio di models_df e ts
    """
    from degiro.degiroUtils import sample_field_parser, get_prediction

    # Genero ss : columns -> indici di ss; ts.columns -> columns di ss
    sample_col = [f"SAMPLE_{sample}" for sample in range(ts.shape[0]-look_back)]
    date_col = [(ts.index[positional:(positional+look_back)]) for positional, date in enumerate(ts.index[:-look_back])]

    columns = []
    for c, sample in enumerate(sample_col):
        for date in date_col[c]:
            columns.append((sample, date))

    ss = pd.DataFrame(index=ts.columns, columns=pd.MultiIndex.from_tuples(columns))
    last_obs = ss.columns.get_level_values(level=1)[-2]
    for i, index in enumerate(ts.index[:len(date_col)]):
        if index >= last_obs:
            logger.debug(index, last_obs)
            break
        try:
            date_indexes = ts.index[i:(i+look_back)]
            cols = ss.columns[ss.columns.get_level_values(level=0)==f"SAMPLE_{i}"]
            ss[cols] = ts.loc[date_indexes].T.astype(np.float)
            
        except:
            logger.debug(i)
            logger.debug(date_indexes)
            logger.debug(cols)

    ss_T = ss.T
    ss_T.index.set_names(["SAMPLE", "DATE"], inplace=True)
    ss_T["Y"] = "GROUND_TRUE"
    ss_T.reset_index("SAMPLE", inplace=True)
    ss_T.rename(columns={"SAMPLE" : "SAMPLE_OLD"}, inplace=True)
    ss_T["SAMPLE"] = ss_T.SAMPLE_OLD.apply(sample_field_parser, look_back=look_back)

    ground_truth = ss_T.reset_index().groupby("SAMPLE_OLD").head(1).iloc[look_back:-1].set_index(["Y", "SAMPLE", "DATE"]).T
    pp = ss.T.reset_index("DATE").iloc[:-look_back*(look_back+1)].set_index("DATE", append=True)
    pp["Y"] = "SAMPLES"
    pp = pp.reset_index().set_index(["Y", "SAMPLE", "DATE"])
    ss = pp.T
    ss = pd.merge(ss, ground_truth, how="inner", left_index=True, right_index=True)

    ground_prediction = pd.DataFrame(index=ss[("GROUND_TRUE")].index, columns=ss[("GROUND_TRUE")].columns, data=None)
    tot_stocks=len(ground_prediction.index)
    for c, isin in enumerate(ground_prediction.index):
        for sample, data in zip(ground_prediction.columns.get_level_values(0), ground_prediction.columns.get_level_values(1)):
            ground_prediction.loc[isin, (sample, data)] = get_prediction(isin, sample, models_df, ss)
        logger.debug(round(c/tot_stocks, 4))

    samples = ss[("SAMPLES")]

    return samples, ground_truth, ground_prediction

In [111]:
%%time
#from degiro.degiroUtils import build_predicted_df

samples, ground_truth, ground_prediction = build_predicted_df(df_4_ref, ts_to_test.iloc[-n_obs:], look_back)

DEGIRO - build_predicted_df - DEBUG - MultiIndex([('SAMPLE_0', '2021-10-22'),
            ('SAMPLE_0', '2021-10-25'),
            ('SAMPLE_0', '2021-11-04'),
            ('SAMPLE_0', '2021-11-06'),
            ('SAMPLE_0', '2021-11-14'),
            ('SAMPLE_0', '2021-11-18'),
            ('SAMPLE_0', '2021-11-22'),
            ('SAMPLE_0', '2021-11-26'),
            ('SAMPLE_0', '2021-12-03'),
            ('SAMPLE_0', '2021-12-04')],
           )
DEGIRO - build_predicted_df - DEBUG - MultiIndex([('SAMPLE_1', '2021-10-25'),
            ('SAMPLE_1', '2021-11-04'),
            ('SAMPLE_1', '2021-11-06'),
            ('SAMPLE_1', '2021-11-14'),
            ('SAMPLE_1', '2021-11-18'),
            ('SAMPLE_1', '2021-11-22'),
            ('SAMPLE_1', '2021-11-26'),
            ('SAMPLE_1', '2021-12-03'),
            ('SAMPLE_1', '2021-12-04'),
            ('SAMPLE_1', '2021-12-10')],
           )
DEGIRO - build_predicted_df - DEBUG - MultiIndex([('SAMPLE_2', '2021-11-04'),
            ('SAMP

ValueError: in user code:

    c:\Users\fontanesio\AppData\Local\Programs\Python\Python37\lib\site-packages\keras\engine\training.py:1586 predict_function  *
        return step_function(self, iterator)
    c:\Users\fontanesio\AppData\Local\Programs\Python\Python37\lib\site-packages\keras\engine\training.py:1576 step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    c:\Users\fontanesio\AppData\Local\Programs\Python\Python37\lib\site-packages\tensorflow\python\distribute\distribute_lib.py:1286 run
        return self._extended.call_for_each_replica(fn, args=args, kwargs=kwargs)
    c:\Users\fontanesio\AppData\Local\Programs\Python\Python37\lib\site-packages\tensorflow\python\distribute\distribute_lib.py:2849 call_for_each_replica
        return self._call_for_each_replica(fn, args, kwargs)
    c:\Users\fontanesio\AppData\Local\Programs\Python\Python37\lib\site-packages\tensorflow\python\distribute\distribute_lib.py:3632 _call_for_each_replica
     

In [38]:
samples.shape, ground_truth.shape, ground_prediction.shape

((4, 1505), (5, 301), (4, 301))

In [44]:
samples.head()

SAMPLE         SAMPLE_0                                              \
DATE         2021-10-01 2021-10-03 2021-10-07 2021-10-08 2021-10-09   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_1                                              \
DATE         2021-10-03 2021-10-07 2021-10-08 2021-10-09 2021-10-11   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_2                                              \
DATE         2021-10-07 2021-10-08 2021-10-09 2021-10-11 2021-10-15   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_3                                              \
DATE         2021-10-08 2021-10-09 2021-10-11 2021-10-15 2021-10-17   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_4                                              \
DATE         2021-10-09 2021-10-11 2021-10-15 2021-10-17 2021-10-21   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_5                                              \
DATE         2021-10-11 2021-10-15 2021-10-17 2021-10-21 2021-10-22   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_6                                              \
DATE         2021-10-15 2021-10-17 2021-10-21 2021-10-22 2021-10-23   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_7                                              \
DATE         2021-10-17 2021-10-21 2021-10-22 2021-10-23 2021-10-25   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_8                                              \
DATE         2021-10-21 2021-10-22 2021-10-23 2021-10-25 2021-10-29   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_9                                              \
DATE         2021-10-22 2021-10-23 2021-10-25 2021-10-29 2021-10-31   
DE000A1EWWW0   

In [45]:
ground_truth.head()

Y            GROUND_TRUE                                              \
SAMPLE          SAMPLE_0   SAMPLE_1   SAMPLE_2   SAMPLE_3   SAMPLE_4   
DATE          2021-10-11 2021-10-15 2021-10-17 2021-10-21 2021-10-22   
SAMPLE_OLD      SAMPLE_5   SAMPLE_6   SAMPLE_7   SAMPLE_8   SAMPLE_9   
DE000A1EWWW0         NaN        NaN        NaN        NaN        NaN   
DE0008404005         NaN        NaN        NaN        NaN        NaN   
DE000BASF111         NaN        NaN        NaN        NaN        NaN   
DE000BAY0017         NaN        NaN        NaN        NaN        NaN   

Y                                                                    \
SAMPLE         SAMPLE_5   SAMPLE_6   SAMPLE_7   SAMPLE_8   SAMPLE_9   
DATE         2021-10-23 2021-10-25 2021-10-29 2021-10-31 2021-11-04   
SAMPLE_OLD    SAMPLE_10  SAMPLE_11  SAMPLE_12  SAMPLE_13  SAMPLE_14   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

Y                                                                    \
SAMPLE        SAMPLE_10  SAMPLE_11  SAMPLE_12  SAMPLE_13  SAMPLE_14   
DATE         2021-11-05 2021-11-06 2021-11-08 2021-11-12 2021-11-14   
SAMPLE_OLD    SAMPLE_15  SAMPLE_16  SAMPLE_17  SAMPLE_18  SAMPLE_19   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

Y                                                                    \
SAMPLE        SAMPLE_15  SAMPLE_16  SAMPLE_17  SAMPLE_18  SAMPLE_19   
DATE         2021-11-18 2021-11-19 2021-11-20 2021-11-22 2021-11-26   
SAMPLE_OLD    SAMPLE_20  SAMPLE_21  SAMPLE_22  SAMPLE_23  SAMPLE_24   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

Y                                                                    \
SAMPLE        SAMPLE_20  SAMPLE_21  SAMPLE_22  SAMPLE_23  SAMPLE_24   
DATE         2021-11-28 2021-12-02 2021-12-03 2021-12-04 2021-12-06   
SAMPLE_OLD    SAMPLE_25  SAMPLE_26  SAMPLE_27  SAMPLE_28  SAMPLE_29   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

Y                                                                    \
SAMPLE        SAMPLE_25  SAMPLE_26  SAMPLE_27  SAMPLE_28  SAMPLE_29   
DATE         2021-12-10 2021-12-12 2021-12-16 2021-12-17 2021-12-18   
SAMPLE_OLD    SAMPLE_30  SAMPLE_31  SAMPLE_32  SAMPLE_33  SAMPLE_34   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

Y                                                                    \
SAMPLE        SAMPLE_30  SAMPLE_31  SAMPLE_32  SAMPLE_33  SAMPLE_34   
DATE         2021-12-20 2021-12-24 2021-12-26 2021-12-30 2021-12-31   
SAMPLE_OLD    SAMPLE_35  SAMPLE_36  SAMPLE_37  SAMPLE_38  SAMPLE_39   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

Y        

In [46]:
ground_prediction.head()

SAMPLE,SAMPLE_0,SAMPLE_1,SAMPLE_2,SAMPLE_3,SAMPLE_4,SAMPLE_5,SAMPLE_6,SAMPLE_7,SAMPLE_8,SAMPLE_9,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22,SAMPLE_23,SAMPLE_24,SAMPLE_25,SAMPLE_26,SAMPLE_27,SAMPLE_28,SAMPLE_29,SAMPLE_30,SAMPLE_31,SAMPLE_32,SAMPLE_33,SAMPLE_34,SAMPLE_35,SAMPLE_36,SAMPLE_37,SAMPLE_38,SAMPLE_39,SAMPLE_40,SAMPLE_41,SAMPLE_42,SAMPLE_43,SAMPLE_44,SAMPLE_45,SAMPLE_46,SAMPLE_47,SAMPLE_48,SAMPLE_49,SAMPLE_50,SAMPLE_51,SAMPLE_52,SAMPLE_53,SAMPLE_54,SAMPLE_55,SAMPLE_56,SAMPLE_57,SAMPLE_58,SAMPLE_59,SAMPLE_60,SAMPLE_61,SAMPLE_62,SAMPLE_63,SAMPLE_64,SAMPLE_65,SAMPLE_66,SAMPLE_67,SAMPLE_68,SAMPLE_69,SAMPLE_70,SAMPLE_71,SAMPLE_72,SAMPLE_73,SAMPLE_74,SAMPLE_75,SAMPLE_76,SAMPLE_77,SAMPLE_78,SAMPLE_79,SAMPLE_80,SAMPLE_81,SAMPLE_82,SAMPLE_83,SAMPLE_84,SAMPLE_85,SAMPLE_86,SAMPLE_87,SAMPLE_88,SAMPLE_89,SAMPLE_90,SAMPLE_91,SAMPLE_92,SAMPLE_93,SAMPLE_94,SAMPLE_95,SAMPLE_96,SAMPLE_97,SAMPLE_98,SAMPLE_99,...,SAMPLE_141,SAMPLE_142,SAMPLE_143,SAMPLE_144,SAMPLE_145,SAMPLE_146,SAMPLE_147,SAMPLE_148,SAMPLE_149,SAMPLE_150,SAMPLE_151,SAMPLE_152,SAMPLE_153,SAMPLE_154,SAMPLE_155,SAMPLE_156,SAMPLE_157,SAMPLE_158,SAMPLE_159,SAMPLE_160,SAMPLE_161,SAMPLE_162,SAMPLE_163,SAMPLE_164,SAMPLE_165,SAMPLE_166,SAMPLE_167,SAMPLE_168,SAMPLE_169,SAMPLE_170,SAMPLE_171,SAMPLE_172,SAMPLE_173,SAMPLE_174,SAMPLE_175,SAMPLE_176,SAMPLE_177,SAMPLE_178,SAMPLE_179,SAMPLE_180,SAMPLE_181,SAMPLE_182,SAMPLE_183,SAMPLE_184,SAMPLE_185,SAMPLE_186,SAMPLE_187,SAMPLE_188,SAMPLE_189,SAMPLE_190,SAMPLE_191,SAMPLE_192,SAMPLE_193,SAMPLE_194,SAMPLE_195,SAMPLE_196,SAMPLE_197,SAMPLE_198,SAMPLE_199,SAMPLE_200,SAMPLE_201,SAMPLE_202,SAMPLE_203,SAMPLE_204,SAMPLE_205,SAMPLE_206,SAMPLE_207,SAMPLE_208,SAMPLE_209,SAMPLE_210,SAMPLE_211,SAMPLE_212,SAMPLE_213,SAMPLE_214,SAMPLE_215,SAMPLE_216,SAMPLE_217,SAMPLE_218,SAMPLE_219,SAMPLE_220,SAMPLE_221,SAMPLE_222,SAMPLE_223,SAMPLE_224,SAMPLE_225,SAMPLE_226,SAMPLE_227,SAMPLE_228,SAMPLE_229,SAMPLE_230,SAMPLE_231,SAMPLE_232,SAMPLE_233,SAMPLE_234,SAMPLE_235,SAMPLE_236,SAMPLE_237,SAMPLE_238,SAMPLE_239,SAMPLE_240
DATE,2021-10-11,2021-10-15,2021-10-17,2021-10-21,2021-10-22,2021-10-23,2021-10-25,2021-10-29,2021-10-31,2021-11-04,2021-11-05,2021-11-06,2021-11-08,2021-11-12,2021-11-14,2021-11-18,2021-11-19,2021-11-20,2021-11-22,2021-11-26,2021-11-28,2021-12-02,2021-12-03,2021-12-04,2021-12-06,2021-12-10,2021-12-12,2021-12-16,2021-12-17,2021-12-18,2021-12-20,2021-12-24,2021-12-26,2021-12-30,2021-12-31,2022-01-01,2022-01-03,2022-01-07,2022-01-09,2022-01-13,2022-01-14,2022-01-15,2022-01-17,2022-01-21,2022-01-23,2022-01-27,2022-01-28,2022-01-30,2022-02-03,2022-02-05,2022-02-07,2022-02-10,2022-02-11,2022-02-13,2022-02-17,2022-02-19,2022-02-20,2022-02-21,2022-02-24,2022-02-26,2022-02-28,2022-03-04,2022-03-05,2022-03-06,2022-03-10,2022-03-12,2022-03-14,2022-03-18,2022-03-19,2022-03-20,2022-03-24,2022-03-26,2022-03-28,2022-04-01,2022-04-02,2022-04-03,2022-04-07,2022-04-09,2022-04-11,2022-04-15,2022-04-16,2022-04-17,2022-04-21,2022-04-23,2022-04-25,2022-04-29,2022-04-30,2022-05-01,2022-05-06,2022-05-08,2022-05-12,2021-05-18,2021-05-21,2021-05-22,2021-05-24,2021-05-28,2021-05-30,2021-06-01,2021-06-04,2021-06-05,...,2021-09-13,2021-09-17,2021-09-19,2021-09-21,2021-09-25,2021-09-26,2021-09-27,2021-10-02,2021-10-04,2021-10-09,2021-10-11,2021-10-12,2021-10-15,2021-10-17,2021-10-19,2021-10-23,2021-10-25,2021-10-26,2021-10-29,2021-10-31,2021-11-02,2021-11-06,2021-11-08,2021-11-09,2021-11-12,2021-11-14,2021-11-16,2021-11-20,2021-11-22,2021-11-23,2021-11-26,2021-11-28,2021-11-30,2021-12-04,2021-12-06,2021-12-07,2021-12-10,2021-12-12,2021-12-14,2021-12-18,2021-12-20,2021-12-21,2021-12-24,2021-12-26,2021-12-28,2022-01-01,2022-01-03,2022-01-04,2022-01-07,2022-01-09,2022-01-11,2022-01-15,2022-01-17,2022-01-18,2022-01-21,2022-01-23,2022-01-25,2022-01-29,2022-01-31,2022-02-01,2022-02-04,2022-02-06,2022-02-08,2022-02-12,2022-02-14,2022-02-15,2022-02-18,2022-02-20,2022-02-22,2022-02-26,2022-02-28,2022-03-0

In [33]:
ground_prediction.reset_index().to_csv(raw_file_path / "ground_prediction.csv")
ground_prediction_test = pd.read_csv(raw_file_path / "ground_prediction.csv", header=[0, 1], index_col=[0, 1])

In [34]:
ground_prediction_test.head()

,SAMPLE,SAMPLE_0,SAMPLE_1,SAMPLE_2,SAMPLE_3,SAMPLE_4,SAMPLE_5,SAMPLE_6,SAMPLE_7,SAMPLE_8,SAMPLE_9,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22,SAMPLE_23,SAMPLE_24,SAMPLE_25,SAMPLE_26,SAMPLE_27,SAMPLE_28,SAMPLE_29,SAMPLE_30,SAMPLE_31,SAMPLE_32,SAMPLE_33,SAMPLE_34,SAMPLE_35,SAMPLE_36,SAMPLE_37,SAMPLE_38,SAMPLE_39,SAMPLE_40,SAMPLE_41,SAMPLE_42,SAMPLE_43,SAMPLE_44,SAMPLE_45,SAMPLE_46,SAMPLE_47,SAMPLE_48,SAMPLE_49,SAMPLE_50,SAMPLE_51,SAMPLE_52,SAMPLE_53,SAMPLE_54,SAMPLE_55,SAMPLE_56,SAMPLE_57,SAMPLE_58,SAMPLE_59,SAMPLE_60,SAMPLE_61,SAMPLE_62,SAMPLE_63,SAMPLE_64,SAMPLE_65,SAMPLE_66,SAMPLE_67,SAMPLE_68,SAMPLE_69,SAMPLE_70,SAMPLE_71,SAMPLE_72,SAMPLE_73,SAMPLE_74,SAMPLE_75,SAMPLE_76,SAMPLE_77,SAMPLE_78,SAMPLE_79,SAMPLE_80,SAMPLE_81,SAMPLE_82,SAMPLE_83,SAMPLE_84,SAMPLE_85,SAMPLE_86,SAMPLE_87,SAMPLE_88,SAMPLE_89,SAMPLE_90,SAMPLE_91,SAMPLE_92,SAMPLE_93,SAMPLE_94,SAMPLE_95,SAMPLE_96,SAMPLE_97,SAMPLE_98,SAMPLE_99,...,SAMPLE_201,SAMPLE_202,SAMPLE_203,SAMPLE_204,SAMPLE_205,SAMPLE_206,SAMPLE_207,SAMPLE_208,SAMPLE_209,SAMPLE_210,SAMPLE_211,SAMPLE_212,SAMPLE_213,SAMPLE_214,SAMPLE_215,SAMPLE_216,SAMPLE_217,SAMPLE_218,SAMPLE_219,SAMPLE_220,SAMPLE_221,SAMPLE_222,SAMPLE_223,SAMPLE_224,SAMPLE_225,SAMPLE_226,SAMPLE_227,SAMPLE_228,SAMPLE_229,SAMPLE_230,SAMPLE_231,SAMPLE_232,SAMPLE_233,SAMPLE_234,SAMPLE_235,SAMPLE_236,SAMPLE_237,SAMPLE_238,SAMPLE_239,SAMPLE_240,SAMPLE_241,SAMPLE_242,SAMPLE_243,SAMPLE_244,SAMPLE_245,SAMPLE_246,SAMPLE_247,SAMPLE_248,SAMPLE_249,SAMPLE_250,SAMPLE_251,SAMPLE_252,SAMPLE_253,SAMPLE_254,SAMPLE_255,SAMPLE_256,SAMPLE_257,SAMPLE_258,SAMPLE_259,SAMPLE_260,SAMPLE_261,SAMPLE_262,SAMPLE_263,SAMPLE_264,SAMPLE_265,SAMPLE_266,SAMPLE_267,SAMPLE_268,SAMPLE_269,SAMPLE_270,SAMPLE_271,SAMPLE_272,SAMPLE_273,SAMPLE_274,SAMPLE_275,SAMPLE_276,SAMPLE_277,SAMPLE_278,SAMPLE_279,SAMPLE_280,SAMPLE_281,SAMPLE_282,SAMPLE_283,SAMPLE_284,SAMPLE_285,SAMPLE_286,SAMPLE_287,SAMPLE_288,SAMPLE_289,SAMPLE_290,SAMPLE_291,SAMPLE_292,SAMPLE_293,SAMPLE_294,SAMPLE_295,SAMPLE_296,SAMPLE_297,SAMPLE_298,SAMPLE_299,SAMPLE_300
,DATE,2021-05-21 00:00:00,2021-05-21 00:00:00,2021-05-24 00:00:00,2021-05-28 00:00:00,2021-05-30 00:00:00,2021-06-03 00:00:00,2021-06-04 00:00:00,2021-06-05 00:00:00,2021-06-11 00:00:00,2021-06-13 00:00:00,2021-06-17 00:00:00,2021-06-19 00:00:00,2021-06-20 00:00:00,2021-06-21 00:00:00,2021-06-25 00:00:00,2021-06-27 00:00:00,2021-07-01 00:00:00,2021-07-02 00:00:00,2021-07-03 00:00:00,2021-07-05 00:00:00,2021-07-09 00:00:00,2021-07-11 00:00:00,2021-07-15 00:00:00,2021-07-16 00:00:00,2021-07-17 00:00:00,2021-07-19 00:00:00,2021-07-23 00:00:00,2021-07-25 00:00:00,2021-07-26 00:00:00,2021-07-29 00:00:00,2021-07-31 00:00:00,2021-08-02 00:00:00,2021-08-06 00:00:00,2021-08-08 00:00:00,2021-08-09 00:00:00,2021-08-12 00:00:00,2021-08-14 00:00:00,2021-08-16 00:00:00,2021-08-20 00:00:00,2021-08-22 00:00:00,2021-08-23 00:00:00,2021-08-26 00:00:00,2021-08-28 00:00:00,2021-08-30 00:00:00,2021-09-03 00:00:00,2021-09-05 00:00:00,2021-09-06 00:00:00,2021-09-09 00:00:00,2021-09-11 00:00:00,2021-09-13 00:00:00,2021-09-17 00:00:00,2021-09-19 00:00:00,2021-09-20 00:00:00,2021-09-24 00:00:00,2021-09-26 00:00:00,2021-10-01 00:00:00,2021-10-03 00:00:00,2021-10-07 00:00:00,2021-10-08 00:00:00,2021-10-09 00:00:00,2021-10-11 00:00:00,2021-10-15 00:00:00,2021-10-17 00:00:00,2021-10-21 00:00:00,2021-10-22 00:00:00,2021-10-23 00:00:00,2021-10-25 00:00:00,2021-10-29 00:00:00,2021-10-31 00:00:00,2021-11-04 00:00:00,2021-11-05 00:00:00,2021-11-06 00:00:00,2021-11-08 00:00:00,2021-11-12 00:00:00,2021-11-14 00:00:00,2021-11-18 00:00:00,2021-11-19 00:00:00,2021-11-20 00:00:00,2021-11-22 00:00:00,2021-11-26 00:00:00,2021-11-28 00:00:00,2021-12-02 00:00:00,2021-12-03 00:00:00,2021-12-04 00:00:00,2021-12-06 00:00:00,2021-12-10 00:00:00,2021-12-12 00:00:00,2021-12-16 00:00:00,2021-12-17 00:00:00,2021-12-18 00:00:00,2021-12-20 00:00:00,2021-12-24 00:00:00,2021-12-26 00:00:00,2021-12-30 00:00:00,2021-12-31 0

In [35]:
samples.head()

SAMPLE         SAMPLE_0                                              \
DATE         2021-05-14 2021-05-15 2021-05-16 2021-05-17 2021-05-18   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_1                                              \
DATE         2021-05-15 2021-05-16 2021-05-17 2021-05-18 2021-05-21   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_2                                              \
DATE         2021-05-16 2021-05-17 2021-05-18 2021-05-21 2021-05-21   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_3                                              \
DATE         2021-05-17 2021-05-18 2021-05-21 2021-05-21 2021-05-24   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_4                                              \
DATE         2021-05-18 2021-05-21 2021-05-21 2021-05-24 2021-05-28   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_5                                              \
DATE         2021-05-21 2021-05-21 2021-05-24 2021-05-28 2021-05-30   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_6                                              \
DATE         2021-05-21 2021-05-24 2021-05-28 2021-05-30 2021-06-03   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_7                                              \
DATE         2021-05-24 2021-05-28 2021-05-30 2021-06-03 2021-06-04   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_8                                              \
DATE         2021-05-28 2021-05-30 2021-06-03 2021-06-04 2021-06-05   
DE000A1EWWW0        NaN        NaN        NaN        NaN        NaN   
DE0008404005        NaN        NaN        NaN        NaN        NaN   
DE000BASF111        NaN        NaN        NaN        NaN        NaN   
DE000BAY0017        NaN        NaN        NaN        NaN        NaN   

SAMPLE         SAMPLE_9                                              \
DATE         2021-05-30 2021-06-03 2021-06-04 2021-06-05 2021-06-11   
DE000A1EWWW0   

In [25]:
sample_col = [f"SAMPLE_{sample}" for sample in range(ts_dax.shape[0]-look_back)]
date_col = [(ts_dax.index[positional:(positional+look_back)]) for positional, date in enumerate(ts_dax.index[:-look_back])]

columns = []
for c, sample in enumerate(sample_col):
    for date in date_col[c]:
        columns.append((sample, date))

ss = pd.DataFrame(index=ts_dax.columns, columns=pd.MultiIndex.from_tuples(columns))
last_obs = ss.columns.get_level_values(level=1)[-2]
for i, index in enumerate(ts_dax.index[:len(date_col)]):
    if index >= last_obs:
        logger.debug(index, last_obs)
        break
    try:
        date_indexes = ts_dax.index[i:(i+look_back)]
        cols = ss.columns[ss.columns.get_level_values(level=0)==f"SAMPLE_{i}"]
        ss[cols] = ts_dax.loc[date_indexes].T.astype(np.float)
        
    except:
        logger.debug(i)
        logger.debug(date_indexes)
        logger.debug(cols)

ss_T = ss.T
ss_T.index.set_names(["SAMPLE", "DATE"], inplace=True)
ss_T["Y"] = "GROUND_TRUE"
ss_T.reset_index("SAMPLE", inplace=True)
ss_T.rename(columns={"SAMPLE" : "SAMPLE_OLD"}, inplace=True)
ss_T["SAMPLE"] = ss_T.SAMPLE_OLD.apply(sample_field_parser, look_back=look_back)

ground_truth = ss_T.reset_index().groupby("SAMPLE_OLD").head(1).iloc[look_back:-1].set_index(["Y", "SAMPLE", "DATE"]).T
pp = ss.T.reset_index("DATE").iloc[:-look_back*(look_back+1)].set_index("DATE", append=True)
pp["Y"] = "SAMPLES"
pp = pp.reset_index().set_index(["Y", "SAMPLE", "DATE"])
ss = pp.T
ss = pd.merge(ss, ground_truth, how="inner", left_index=True, right_index=True)

ground_prediction = pd.DataFrame(index=ss[("GROUND_TRUE")].index, columns=ss[("GROUND_TRUE")].columns, data=None)
tot_stocks=len(ground_prediction.index)
for c, isin in enumerate(ground_prediction.index):
    for sample, data in zip(ground_prediction.columns.get_level_values(0), ground_prediction.columns.get_level_values(1)):
        ground_prediction.loc[isin, (sample, data)] = get_prediction(isin, sample, rnn_models, ss)
    logger.debug(round(c/tot_stocks, 4))

samples = ss[("SAMPLES")]

samples, ground_truth, ground_prediction

In [ ]:
ss.reset_index().to_csv("C:\\Users\\fontanesio\\Documents\\pythonScripts\\progetti\\personali\\laboratorio\\file\\raw\\dax_ground_true.csv", sep=";")

In [35]:
from degiro.degiroUtils import sample_field_parser, get_prediction

In [29]:
ss_T = ss.T
ss_T.index.set_names(["SAMPLE", "DATE"], inplace=True)
ss_T["Y"] = "GROUND_TRUE"
ss_T.reset_index("SAMPLE", inplace=True)
ss_T.rename(columns={"SAMPLE" : "SAMPLE_OLD"}, inplace=True)
ss_T["SAMPLE"] = ss_T.SAMPLE_OLD.apply(sample_field_parser, look_back=look_back)

ground_truth = ss_T.reset_index().groupby("SAMPLE_OLD").head(1).iloc[look_back:-1].set_index(["Y", "SAMPLE", "DATE"]).T
pp = ss.T.reset_index("DATE").iloc[:-look_back*(look_back+1)].set_index("DATE", append=True)
pp["Y"] = "SAMPLES"
pp = pp.reset_index().set_index(["Y", "SAMPLE", "DATE"])
ss = pp.T
ss = pd.merge(ss, ground_truth, how="inner", left_index=True, right_index=True)

Y               SAMPLES                                              \
SAMPLE         SAMPLE_0   SAMPLE_1   SAMPLE_2   SAMPLE_3   SAMPLE_4   
DATE         2019-03-07 2019-03-08 2019-03-09 2019-03-10 2019-03-13   
DE000A1EWWW0      279.2      289.5      290.2      294.2      288.9   
DE0008404005      210.0      208.4      206.8      203.0      199.8   
DE000BASF111      71.08      71.14      70.27      69.09      67.73   
DE000BAY0017      51.84      51.62      52.06      52.34      50.12   
DE0005190003      76.29       76.1      72.56       72.1       71.5   

Y                                                                    ...  \
SAMPLE         SAMPLE_5   SAMPLE_6   SAMPLE_7   SAMPLE_8   SAMPLE_9  ...   
DATE         2019-03-14 2019-03-15 2019-03-16 2019-03-17 2019-03-20  ...   
DE000A1EWWW0      293.1      288.7      290.6      291.1      295.6  ...   
DE0008404005      202.0     200.45     198.26     194.92     195.38  ...   
DE000BASF111      68.16       69.4      68.32      68.99       68.0  ...   
DE000BAY0017      51.59      55.11      53.34      53.77      53.91  ...   
DE0005190003       71.5      70.39      69.32      71.27      71.58  ...   

Y            GROUND_TRUE                                              \
SAMPLE        SAMPLE_240 SAMPLE_241 SAMPLE_242 SAMPLE_243 SAMPLE_244   
DATE          2020-02-19 2020-02-20 2020-02-21 2020-02-22 2020-02-23   
DE000A1EWWW0       191.5     177.08     166.92      173.0      175.7   
DE0008404005      134.74      122.1      119.0     129.92     132.42   
DE000BASF111      40.795     40.205      39.29      41.06     39.035   
DE000BAY0017        50.1     49.815       48.1      50.57     48.125   
DE0005190003       41.37      37.66     39.135     40.355      40.09   

Y                                                                    
SAMPLE       SAMPLE_245 SAMPLE_246 SAMPLE_247 SAMPLE_248 SAMPLE_249  
DATE         2020-02-26 2020-02-27 2020-02-28 2020-02-29 2020-03-01  
DE000A1EWWW0     171.94     173.88      200.8      221.0        NaN  
DE0008404005     146.92      146.5     173.02      173.3        NaN  
DE000BASF111      41.25      40.91      46.38      46.21        NaN  
DE000BAY0017      48.05     48.205      56.07      56.44        NaN  
DE0005190003      45.42      44.08      50.65      50.33        NaN  

[5 rows x 500 columns]

In [30]:
ss.drop(columns=[ss.columns[-1]], inplace=True)

In [36]:
ground_prediction = pd.DataFrame(index=ss[("GROUND_TRUE")].index, columns=ss[("GROUND_TRUE")].columns, data=None)
tot_stocks=len(ground_prediction.index)
for c, isin in enumerate(ground_prediction.index):
    for sample, data in zip(ground_prediction.columns.get_level_values(0), ground_prediction.columns.get_level_values(1)):
        ground_prediction.loc[isin, (sample, data)] = get_prediction(isin, sample, rnn_models, ss)
    logger.debug(round(c/tot_stocks, 4))

0.0
0.0333
0.0667
0.1
0.1333
0.1667
0.2
0.2333
0.2667
0.3
0.3333
0.3667
0.4
0.4333
0.4667
0.5
0.5333
0.5667
0.6
0.6333
0.6667
0.7
0.7333
0.7667
0.8
0.8333
0.8667
0.9
0.9333
0.9667


## TEST CHECKPOINTS : SAVING AND LOADING FILES
    dax_anagrafica.parquet -> DIFF
    ts_dax.parquet -> EQUALS                    -> OK
    dax_models.parquet -> TYPE DIFF             -> OK

In [90]:
from degiro.degiroUtils import check_equality_btwn_df

In [91]:
"""
dax.P_HISTORICAL.iloc[0]
'expires': '2021-03-04T18:41:29.1355696+01:00'
ultimo -> [364, 279.2]] Ã¨ di oggi 04/03/2020
"""

"\ndax.P_HISTORICAL.iloc[0]\n'expires': '2021-03-04T18:41:29.1355696+01:00'\nultimo -> [364, 279.2]] Ã¨ di oggi 04/03/2020\n"

In [92]:
dax.head()

,P_REAL_TIME,P_HISTORICAL,ANAGRAFICA,NAME,ISIN,SYMBOL,CONTRACT_SIZE,TRADEBLE,P_CLOSE,P_VECTOR,LOG_RET_VECTOR,20_D_AVG,20_D_STD,10_D_AVG,10_D_STD,R_CUMSUM,DAILY_THRESHOLD,DAILY_THRESHOLD_COUNTER,DAILY_THRESHOLD_INT,TIME_SERIES
3457,{'expires': '2021-03-04T18:41:29.1345705+01:00...,"{'times': '2020-03-05/P1D', 'expires': '2021-0...","{'id': '3457', 'name': 'Adidas AG', 'isin': 'D...",Adidas AG,DE000A1EWWW0,ADS,1.0,True,289.50,"[249.1, 241.05, 227.3, 221.0, 200.8, 173.88, 1...","[-0.03285004141650339, -0.05873364999313381, -...",0.001487,0.022197,0.001882,0.023856,1.013092,"[SELL, SELL, SELL, SELL, SELL, HOLD AND PRAY, ...","{'SELL': 31, 'HOLD AND PRAY': 185, 'BUY': 36}","[-1, -1, -1, -1, -1, 0, 0, 0, -1, 1, 1, -1, 1,...",DE000A1EWWW0 DATE ...
4799,{'expires': '2021-03-04T18:41:29.8636333+01:00...,"{'times': '2020-03-05/P1D', 'expires': '2021-0...","{'id': '4799', 'name': 'Allianz SE', 'isin': '...",Allianz SE,DE0008404005,ALV,1.0,True,208.40,"[195.0, 188.9, 171.7, 173.3, 173.02, 146.5, 14...","[-0.03178178405628296, -0.09546900660403405, 0...",0.001409,0.019984,0.002241,0.022868,1.070587,"[SELL, SELL, HOLD AND PRAY, HOLD AND PRAY, SEL...","{'SELL': 22, 'HOLD AND PRAY': 202, 'BUY': 28}","[-1, -1, 0, 0, -1, 0, -1, 0, -1, 1, 1, 0, 1, 1...",DE0008404005 DATE ...
3572,{'expires': '2021-03-04T18:41:30.5036935+01:00...,"{'times': '2020-03-05/P1D', 'expires': '2021-0...","{'id': '3572', 'name': 'BASF SE', 'isin': 'DE0...",BASF SE,DE000BASF111,BAS,1.0,True,71.14,"[53.32, 52.67, 47.105, 46.21, 46.38, 40.91, 41...","[-0.01226546181520991, -0.11166688099466696, -...",0.002311,0.023429,0.002355,0.024455,1.065141,"[HOLD AND PRAY, SELL, HOLD AND PRAY, HOLD AND ...","{'HOLD AND PRAY': 194, 'SELL': 26, 'BUY': 32}","[0, -1, 0, 0, -1, 0, -1, 1, -1, 0, 0, -1, 1, 0...",DE000BASF111 DATE ...
3934,{'expires': '2021-03-04T18:41:31.1607454+01:00...,"{'times': '2020-03-05/P1D', 'expires': '2021-0...","{'id': '3934', 'name': 'Bayer AG', 'isin': 'DE...",Bayer AG,DE000BAY0017,BAYN,1.0,True,51.62,"[66.33, 63.78, 58.42, 56.44, 56.07, 48.205, 48...","[-0.03920262195555299, -0.08778136462231666, -...",-0.000083,0.024336,0.000165,0.024850,0.950371,"[SELL, SELL, SELL, HOLD AND PRAY, SELL, HOLD A...","{'SELL': 31, 'HOLD AND PRAY': 188, 'BUY': 33}","[-1, -1, -1, 0, -1, 0, 0, 1, -1, 1, 0, -1, 1, ...",DE000BAY0017 DATE ...
3597,{'expires': '2021-03-04T18:41:31.8388167+01:00...,"{'times': '2020-03-05/P1D', 'expires': '2021-0...","{'id': '3597', 'name': 'Bayerische Motoren Wer...",Bayerische Motoren Werke AG,DE0005190003,BMW,1.0,True,76.10,"[58.07, 57.22, 51.02, 50.33, 50.65, 44.08, 45....","[-0.014745691761994301, -0.11468577490829877, ...",0.002288,0.022271,0.002917,0.024871,1.075877,"[HOLD AND PRAY, SELL, HOLD AND PRAY, HOLD AND ...","{'HOLD AND PRAY': 183, 'SELL': 31, 'BUY': 38}","[0, -1, 0, 0, -1, 1, -1, 0, -1, -1, 1, -1, 1, ...",DE0005190003 DATE ...


### export / load dax_anagrafica.parquet 

In [130]:
dax_anagrafica = anagrafica_builder(dax, "ANAGRAFICA")
dax_anagrafica.set_index("isin", inplace=True)
dax_anagrafica.head()
dax_anagrafica.to_parquet(raw_file_path / "dax_anagrafica.parquet")

In [148]:
dax_anagrafica_test = pd.read_parquet(raw_file_path / "dax_anagrafica.parquet")
check_equality_btwn_df(dax_anagrafica, dax_anagrafica_test)

DFS DIFFERSS
EQUALS id
EQUALS name
EQUALS symbol
-------------> contractSize
-------------> 	(dtype('O'), dtype('float64'))
EQUALS productType
-------------> productTypeId
-------------> 	(dtype('O'), dtype('int64'))
-------------> tradable
-------------> 	(dtype('O'), dtype('bool'))
EQUALS category
EQUALS currency
-------------> strikePrice
-------------> 	(dtype('O'), dtype('float64'))
EQUALS exchangeId
-------------> onlyEodPrices
-------------> 	(dtype('O'), dtype('bool'))
-------------> orderTimeTypes
-------------> 	(dtype('O'), dtype('O'))
-------------> buyOrderTypes
-------------> 	(dtype('O'), dtype('O'))
-------------> sellOrderTypes
-------------> 	(dtype('O'), dtype('O'))
-------------> productBitTypes
-------------> 	(dtype('O'), dtype('O'))
-------------> closePrice
-------------> 	(dtype('O'), dtype('float64'))
EQUALS closePriceDate
EQUALS feedQuality
-------------> orderBookDepth
-------------> 	(dtype('O'), dtype('int64'))
EQUALS vwdIdentifierType
EQUALS vwdId
-------

### export / load ts_dax.parquet

In [143]:
ts_dax.to_parquet(raw_file_path / "ts_dax.parquet", engine="fastparquet")

In [149]:
check_equality_btwn_df(ts_dax, pd.read_parquet(raw_file_path / "ts_dax_test.parquet"))

DFS ARE EQUALS


### export / load dax_models.parquet

In [118]:
from degiro.degiroUtils import rnn_load
rnn_models_test = pd.read_parquet(path=raw_file_path / "dax_models.parquet", engine="fastparquet").set_index("index")
rnn_models_test.index.name = None
rnn_models_test["MODEL"] = rnn_models_test.apply(lambda x: rnn_load(path=_MODEL_PATH_ / x.name), axis=1)

In [150]:
check_equality_btwn_df(rnn_models.drop(columns=["PREDICTED_Y", "TEST_Y"]), rnn_models_test)

DFS DIFFERSS
	SAME TYPE MODEL -> <class 'tensorflow.python.keras.engine.sequential.Sequential'> <class 'tensorflow.python.keras.engine.sequential.Sequential'>
	SAME TYPE SCALER_MIN -> <class 'numpy.float64'> <class 'numpy.float64'>
	SAME TYPE SCALER_MAX -> <class 'numpy.float64'> <class 'numpy.float64'>
EQUALS TRAIN_LOSS
EQUALS TEST_LOSS
EQUALS VOL_IS
EQUALS VOL_OOS
EQUALS TIMESTAMP
EQUALS AVERAGE_PRICE
EQUALS LOOK_BACK
EQUALS LAST_PRICE
EQUALS TRAIN_LOSS_PERC
EQUALS TEST_LOSS_PERC
EQUALS VOL_IS_PERC
EQUALS VOL_OOS_PERC


### export / ground_prediction.csv, ground_true, samples

In [187]:
ground_prediction.reset_index().to_csv(raw_file_path / "ground_prediction.csv")
ground_prediction_test = pd.read_csv(raw_file_path / "ground_prediction.csv", header=[0, 1], index_col=[0, 1])

In [188]:
ground_prediction.shape

(30, 249)

In [189]:
ground_prediction_test.reset_index(level=0, drop=True).shape

(30, 249)

In [42]:
ground_true = ss[("GROUND_TRUE")]
ground_true.head()

SAMPLE,SAMPLE_0,SAMPLE_1,SAMPLE_2,SAMPLE_3,SAMPLE_4,SAMPLE_5,SAMPLE_6,SAMPLE_7,SAMPLE_8,SAMPLE_9,...,SAMPLE_239,SAMPLE_240,SAMPLE_241,SAMPLE_242,SAMPLE_243,SAMPLE_244,SAMPLE_245,SAMPLE_246,SAMPLE_247,SAMPLE_248
DATE,2019-03-08,2019-03-09,2019-03-10,2019-03-13,2019-03-14,2019-03-15,2019-03-16,2019-03-17,2019-03-20,2019-03-21,...,2020-02-16,2020-02-19,2020-02-20,2020-02-21,2020-02-22,2020-02-23,2020-02-26,2020-02-27,2020-02-28,2020-02-29
DE000A1EWWW0,289.5,290.2,294.2,288.9,293.1,288.7,290.6,291.1,295.6,288.8,...,180.62,191.5,177.08,166.92,173.0,175.7,171.94,173.88,200.8,221.0
DE0008404005,208.4,206.8,203.0,199.8,202.0,200.45,198.26,194.92,195.38,194.86,...,131.74,134.74,122.1,119.0,129.92,132.42,146.92,146.5,173.02,173.3
DE000BASF111,71.14,70.27,69.09,67.73,68.16,69.4,68.32,68.99,68.0,67.03,...,39.635,40.795,40.205,39.29,41.06,39.035,41.25,40.91,46.38,46.21
DE000BAY0017,51.62,52.06,52.34,50.12,51.59,55.11,53.34,53.77,53.91,53.11,...,47.5,50.1,49.815,48.1,50.57,48.125,48.05,48.205,56.07,56.44
DE0005190003,76.1,72.56,72.1,71.5,71.5,70.39,69.32,71.27,71.58,71.03,...,39.8,41.37,37.66,39.135,40.355,40.09,45.42,44.08,50.65,50.33


In [43]:
samples = ss[("SAMPLES")]
samples.head()

SAMPLE,SAMPLE_0,SAMPLE_1,SAMPLE_2,SAMPLE_3,SAMPLE_4,SAMPLE_5,SAMPLE_6,SAMPLE_7,SAMPLE_8,SAMPLE_9,...,SAMPLE_240,SAMPLE_241,SAMPLE_242,SAMPLE_243,SAMPLE_244,SAMPLE_245,SAMPLE_246,SAMPLE_247,SAMPLE_248,SAMPLE_249
DATE,2019-03-07,2019-03-08,2019-03-09,2019-03-10,2019-03-13,2019-03-14,2019-03-15,2019-03-16,2019-03-17,2019-03-20,...,2020-02-16,2020-02-19,2020-02-20,2020-02-21,2020-02-22,2020-02-23,2020-02-26,2020-02-27,2020-02-28,2020-02-29
DE000A1EWWW0,279.2,289.5,290.2,294.2,288.9,293.1,288.7,290.6,291.1,295.6,...,180.62,191.5,177.08,166.92,173.0,175.7,171.94,173.88,200.8,221.0
DE0008404005,210.0,208.4,206.8,203.0,199.8,202.0,200.45,198.26,194.92,195.38,...,131.74,134.74,122.1,119.0,129.92,132.42,146.92,146.5,173.02,173.3
DE000BASF111,71.08,71.14,70.27,69.09,67.73,68.16,69.4,68.32,68.99,68.0,...,39.635,40.795,40.205,39.29,41.06,39.035,41.25,40.91,46.38,46.21
DE000BAY0017,51.84,51.62,52.06,52.34,50.12,51.59,55.11,53.34,53.77,53.91,...,47.5,50.1,49.815,48.1,50.57,48.125,48.05,48.205,56.07,56.44
DE0005190003,76.29,76.1,72.56,72.1,71.5,71.5,70.39,69.32,71.27,71.58,...,39.8,41.37,37.66,39.135,40.355,40.09,45.42,44.08,50.65,50.33


In [34]:
samples.head()

SAMPLE,SAMPLE_0,SAMPLE_1,SAMPLE_2,SAMPLE_3,SAMPLE_4,SAMPLE_5,SAMPLE_6,SAMPLE_7,SAMPLE_8,SAMPLE_9,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22,SAMPLE_23,SAMPLE_24,SAMPLE_25,SAMPLE_26,SAMPLE_27,SAMPLE_28,SAMPLE_29,SAMPLE_30,SAMPLE_31,SAMPLE_32,SAMPLE_33,SAMPLE_34,SAMPLE_35,SAMPLE_36,SAMPLE_37,SAMPLE_38,SAMPLE_39,SAMPLE_40,SAMPLE_41,SAMPLE_42,SAMPLE_43,SAMPLE_44,SAMPLE_45,SAMPLE_46,SAMPLE_47,SAMPLE_48,SAMPLE_49,SAMPLE_50,SAMPLE_51,SAMPLE_52,SAMPLE_53,SAMPLE_54,SAMPLE_55,SAMPLE_56,SAMPLE_57,SAMPLE_58,SAMPLE_59,SAMPLE_60,SAMPLE_61,SAMPLE_62,SAMPLE_63,SAMPLE_64,SAMPLE_65,SAMPLE_66,SAMPLE_67,SAMPLE_68,SAMPLE_69,SAMPLE_70,SAMPLE_71,SAMPLE_72,SAMPLE_73,SAMPLE_74,SAMPLE_75,SAMPLE_76,SAMPLE_77,SAMPLE_78,SAMPLE_79,SAMPLE_80,SAMPLE_81,SAMPLE_82,SAMPLE_83,SAMPLE_84,SAMPLE_85,SAMPLE_86,SAMPLE_87,SAMPLE_88,SAMPLE_89,SAMPLE_90,SAMPLE_91,SAMPLE_92,SAMPLE_93,SAMPLE_94,SAMPLE_95,SAMPLE_96,SAMPLE_97,SAMPLE_98,SAMPLE_99,...,SAMPLE_132,SAMPLE_133,SAMPLE_134,SAMPLE_135,SAMPLE_136,SAMPLE_137,SAMPLE_138,SAMPLE_139,SAMPLE_140,SAMPLE_141,SAMPLE_142,SAMPLE_143,SAMPLE_144,SAMPLE_145,SAMPLE_146,SAMPLE_147,SAMPLE_148,SAMPLE_149,SAMPLE_150,SAMPLE_151,SAMPLE_152,SAMPLE_153,SAMPLE_154,SAMPLE_155,SAMPLE_156,SAMPLE_157,SAMPLE_158,SAMPLE_159,SAMPLE_160,SAMPLE_161,SAMPLE_162,SAMPLE_163,SAMPLE_164,SAMPLE_165,SAMPLE_166,SAMPLE_167,SAMPLE_168,SAMPLE_169,SAMPLE_170,SAMPLE_171,SAMPLE_172,SAMPLE_173,SAMPLE_174,SAMPLE_175,SAMPLE_176,SAMPLE_177,SAMPLE_178,SAMPLE_179,SAMPLE_180,SAMPLE_181,SAMPLE_182,SAMPLE_183,SAMPLE_184,SAMPLE_185,SAMPLE_186,SAMPLE_187,SAMPLE_188,SAMPLE_189,SAMPLE_190,SAMPLE_191,SAMPLE_192,SAMPLE_193,SAMPLE_194,SAMPLE_195,SAMPLE_196,SAMPLE_197,SAMPLE_198,SAMPLE_199,SAMPLE_200,SAMPLE_201,SAMPLE_202,SAMPLE_203,SAMPLE_204,SAMPLE_205,SAMPLE_206,SAMPLE_207,SAMPLE_208,SAMPLE_209,SAMPLE_210,SAMPLE_211,SAMPLE_212,SAMPLE_213,SAMPLE_214,SAMPLE_215,SAMPLE_216,SAMPLE_217,SAMPLE_218,SAMPLE_219,SAMPLE_220,SAMPLE_221,SAMPLE_222,SAMPLE_223,SAMPLE_224,SAMPLE_225,SAMPLE_226,SAMPLE_227,SAMPLE_228,SAMPLE_229,SAMPLE_230,SAMPLE_231
DATE,2019-12-04,2019-12-05,2019-12-06,2019-12-09,2019-12-10,2019-12-11,2019-12-12,2019-12-13,2019-12-16,2019-12-17,2019-12-18,2019-12-19,2019-12-20,2019-12-23,2019-12-24,2019-12-25,2019-12-26,2019-12-27,2019-12-30,2019-12-31,2020-01-01,2020-01-02,2020-01-03,2020-01-06,2020-01-07,2020-01-08,2020-01-09,2020-01-10,2020-01-13,2020-01-14,2020-01-15,2020-01-16,2020-01-17,2020-01-20,2020-01-21,2020-01-22,2020-01-23,2020-01-24,2020-01-27,2020-01-28,2020-01-29,2020-01-30,2020-01-31,2020-02-03,2020-02-04,2020-02-05,2020-02-06,2020-02-07,2020-02-10,2020-02-11,2020-02-12,2020-02-13,2020-02-14,2020-02-17,2020-02-18,2020-02-19,2020-02-20,2020-02-21,2020-02-24,2020-02-25,2020-02-26,2020-02-27,2020-02-28,2020-03-02,2020-03-03,2020-03-04,2020-03-05,2020-03-06,2020-03-09,2020-03-10,2020-03-11,2020-03-12,2020-03-13,2020-03-16,2020-03-17,2020-03-18,2020-03-19,2020-03-20,2020-03-23,2020-03-24,2020-03-25,2020-03-26,2020-03-27,2020-03-30,2020-03-31,2020-04-01,2020-04-02,2020-04-03,2020-04-06,2020-04-07,2020-04-08,2020-04-09,2020-04-10,2020-04-13,2020-04-14,2020-04-15,2020-04-16,2020-04-17,2020-04-20,2020-04-21,...,2020-06-05,2020-06-08,2020-06-09,2020-06-10,2020-06-11,2020-06-15,2020-06-16,2020-06-17,2020-06-18,2020-06-19,2020-06-22,2020-06-23,2020-06-24,2020-06-25,2020-06-26,2020-06-29,2020-06-30,2020-07-01,2020-07-02,2020-07-03,2020-07-06,2020-07-07,2020-07-08,2020-07-09,2020-07-10,2020-07-13,2020-07-14,2020-07-15,2020-07-16,2020-07-17,2020-07-20,2020-07-21,2020-07-22,2020-07-23,2020-07-24,2020-07-27,2020-07-28,2020-07-29,2020-07-30,2020-08-04,2020-08-05,2020-08-06,2020-08-07,2020-08-10,2020-08-11,2020-08-12,2020-08-13,2020-08-14,2020-08-17,2020-08-18,2020-08-19,2020-08-20,2020-08-21,2020-08-24,2020-08-25,2020-08-26,2020-08-27,2020-08-28,2020-08-31,2020-09-01,2020-09-02,2020-09-03,2020-09-04,2020-09-07,2020-09-08,2020-09-09,2020-09-10,2020-09-11,2020-09-14,2020-09-15,2020-09-16,2020-09-1

In [41]:
ground_truth.head()

Y            GROUND_TRUE                                              \
SAMPLE          SAMPLE_0   SAMPLE_1   SAMPLE_2   SAMPLE_3   SAMPLE_4   
DATE          2019-12-05 2019-12-06 2019-12-09 2019-12-10 2019-12-11   
SAMPLE_OLD      SAMPLE_1   SAMPLE_2   SAMPLE_3   SAMPLE_4   SAMPLE_5   
DE000A1EWWW0       256.2      254.5    252.475      268.4      268.4   
DE0008404005       192.7     192.32     191.44     202.45      203.8   
DE000BASF111       57.93      57.93      57.86      61.62      61.98   
DE000BAY0017       44.65     45.325     45.535     47.465    47.5025   

Y                                                                    \
SAMPLE         SAMPLE_5   SAMPLE_6   SAMPLE_7   SAMPLE_8   SAMPLE_9   
DATE         2019-12-12 2019-12-13 2019-12-16 2019-12-17 2019-12-18   
SAMPLE_OLD     SAMPLE_6   SAMPLE_7   SAMPLE_8   SAMPLE_9  SAMPLE_10   
DE000A1EWWW0     272.85      279.6     282.35      282.5    284.125   
DE0008404005    203.575     203.25     202.75     204.75     205.25   
DE000BASF111      63.01      61.93      61.61      62.66     62.735   
DE000BAY0017      48.93      48.29      48.18      49.31    49.9025   

Y                                                                    \
SAMPLE        SAMPLE_10  SAMPLE_11  SAMPLE_12  SAMPLE_13  SAMPLE_14   
DATE         2019-12-19 2019-12-20 2019-12-23 2019-12-24 2019-12-25   
SAMPLE_OLD    SAMPLE_11  SAMPLE_12  SAMPLE_13  SAMPLE_14  SAMPLE_15   
DE000A1EWWW0      286.5      281.9      280.6      281.1      285.3   
DE0008404005      205.8     205.75    205.075     205.15     205.55   
DE000BASF111       62.9      62.21      62.66      62.35       62.0   
DE000BAY0017      50.24     50.415      50.93      50.95      51.33   

Y                                                                    \
SAMPLE        SAMPLE_15  SAMPLE_16  SAMPLE_17  SAMPLE_18  SAMPLE_19   
DATE         2019-12-26 2019-12-27 2019-12-30 2019-12-31 2020-01-01   
SAMPLE_OLD    SAMPLE_16  SAMPLE_17  SAMPLE_18  SAMPLE_19  SAMPLE_20   
DE000A1EWWW0      295.5      294.0     298.55      294.1     291.85   
DE0008404005      203.3     203.15     203.45     201.55      203.1   
DE000BASF111      61.73     62.125       62.0      61.53       62.5   
DE000BAY0017      50.77      50.15      49.65      49.97      50.24   

Y                                                                    \
SAMPLE        SAMPLE_20  SAMPLE_21  SAMPLE_22  SAMPLE_23  SAMPLE_24   
DATE         2020-01-02 2020-01-03 2020-01-06 2020-01-07 2020-01-08   
SAMPLE_OLD    SAMPLE_21  SAMPLE_22  SAMPLE_23  SAMPLE_24  SAMPLE_25   
DE000A1EWWW0      285.7    285.375      283.3     284.05     279.15   
DE0008404005      202.8      202.7     201.15     200.95      201.2   
DE000BASF111      62.07      62.97     62.265      62.56      63.53   
DE000BAY0017      50.03    49.4925     48.725      48.71      48.51   

Y                                                                    \
SAMPLE        SAMPLE_25  SAMPLE_26  SAMPLE_27  SAMPLE_28  SAMPLE_29   
DATE         2020-01-09 2020-01-10 2020-01-13 2020-01-14 2020-01-15   
SAMPLE_OLD    SAMPLE_26  SAMPLE_27  SAMPLE_28  SAMPLE_29  SAMPLE_30   
DE000A1EWWW0      280.2      278.0      278.0     275.55      269.0   
DE0008404005      201.3     198.54      199.0     198.64     199.24   
DE000BASF111      63.73      63.32      62.89      62.86      63.48   
DE000BAY0017     48.825      48.56      48.31     48.105     48.155   

Y                                                                    \
SAMPLE        SAMPLE_30  SAMPLE_31  SAMPLE_32  SAMPLE_33  SAMPLE_34   
DATE         2020-01-16 2020-01-17 2020-01-20 2020-01-21 2020-01-22   
SAMPLE_OLD    SAMPLE_31  SAMPLE_32  SAMPLE_33  SAMPLE_34  SAMPLE_35   
DE000A1EWWW0     268.35     270.65      274.4      267.0      263.0   
DE0008404005     199.08     198.54      198.6      196.4     195.08   
DE000BASF111     63.885      64.07      64.95      65.22      65.01   
DE000BAY0017      47.29      46.63    48.0375      47.78      47.08   

Y        

In [42]:
ground_prediction.head()

SAMPLE,SAMPLE_0,SAMPLE_1,SAMPLE_2,SAMPLE_3,SAMPLE_4,SAMPLE_5,SAMPLE_6,SAMPLE_7,SAMPLE_8,SAMPLE_9,SAMPLE_10,SAMPLE_11,SAMPLE_12,SAMPLE_13,SAMPLE_14,SAMPLE_15,SAMPLE_16,SAMPLE_17,SAMPLE_18,SAMPLE_19,SAMPLE_20,SAMPLE_21,SAMPLE_22,SAMPLE_23,SAMPLE_24,SAMPLE_25,SAMPLE_26,SAMPLE_27,SAMPLE_28,SAMPLE_29,SAMPLE_30,SAMPLE_31,SAMPLE_32,SAMPLE_33,SAMPLE_34,SAMPLE_35,SAMPLE_36,SAMPLE_37,SAMPLE_38,SAMPLE_39,SAMPLE_40,SAMPLE_41,SAMPLE_42,SAMPLE_43,SAMPLE_44,SAMPLE_45,SAMPLE_46,SAMPLE_47,SAMPLE_48,SAMPLE_49,SAMPLE_50,SAMPLE_51,SAMPLE_52,SAMPLE_53,SAMPLE_54,SAMPLE_55,SAMPLE_56,SAMPLE_57,SAMPLE_58,SAMPLE_59,SAMPLE_60,SAMPLE_61,SAMPLE_62,SAMPLE_63,SAMPLE_64,SAMPLE_65,SAMPLE_66,SAMPLE_67,SAMPLE_68,SAMPLE_69,SAMPLE_70,SAMPLE_71,SAMPLE_72,SAMPLE_73,SAMPLE_74,SAMPLE_75,SAMPLE_76,SAMPLE_77,SAMPLE_78,SAMPLE_79,SAMPLE_80,SAMPLE_81,SAMPLE_82,SAMPLE_83,SAMPLE_84,SAMPLE_85,SAMPLE_86,SAMPLE_87,SAMPLE_88,SAMPLE_89,SAMPLE_90,SAMPLE_91,SAMPLE_92,SAMPLE_93,SAMPLE_94,SAMPLE_95,SAMPLE_96,SAMPLE_97,SAMPLE_98,SAMPLE_99,...,SAMPLE_132,SAMPLE_133,SAMPLE_134,SAMPLE_135,SAMPLE_136,SAMPLE_137,SAMPLE_138,SAMPLE_139,SAMPLE_140,SAMPLE_141,SAMPLE_142,SAMPLE_143,SAMPLE_144,SAMPLE_145,SAMPLE_146,SAMPLE_147,SAMPLE_148,SAMPLE_149,SAMPLE_150,SAMPLE_151,SAMPLE_152,SAMPLE_153,SAMPLE_154,SAMPLE_155,SAMPLE_156,SAMPLE_157,SAMPLE_158,SAMPLE_159,SAMPLE_160,SAMPLE_161,SAMPLE_162,SAMPLE_163,SAMPLE_164,SAMPLE_165,SAMPLE_166,SAMPLE_167,SAMPLE_168,SAMPLE_169,SAMPLE_170,SAMPLE_171,SAMPLE_172,SAMPLE_173,SAMPLE_174,SAMPLE_175,SAMPLE_176,SAMPLE_177,SAMPLE_178,SAMPLE_179,SAMPLE_180,SAMPLE_181,SAMPLE_182,SAMPLE_183,SAMPLE_184,SAMPLE_185,SAMPLE_186,SAMPLE_187,SAMPLE_188,SAMPLE_189,SAMPLE_190,SAMPLE_191,SAMPLE_192,SAMPLE_193,SAMPLE_194,SAMPLE_195,SAMPLE_196,SAMPLE_197,SAMPLE_198,SAMPLE_199,SAMPLE_200,SAMPLE_201,SAMPLE_202,SAMPLE_203,SAMPLE_204,SAMPLE_205,SAMPLE_206,SAMPLE_207,SAMPLE_208,SAMPLE_209,SAMPLE_210,SAMPLE_211,SAMPLE_212,SAMPLE_213,SAMPLE_214,SAMPLE_215,SAMPLE_216,SAMPLE_217,SAMPLE_218,SAMPLE_219,SAMPLE_220,SAMPLE_221,SAMPLE_222,SAMPLE_223,SAMPLE_224,SAMPLE_225,SAMPLE_226,SAMPLE_227,SAMPLE_228,SAMPLE_229,SAMPLE_230,SAMPLE_231
DATE,2019-12-05,2019-12-06,2019-12-09,2019-12-10,2019-12-11,2019-12-12,2019-12-13,2019-12-16,2019-12-17,2019-12-18,2019-12-19,2019-12-20,2019-12-23,2019-12-24,2019-12-25,2019-12-26,2019-12-27,2019-12-30,2019-12-31,2020-01-01,2020-01-02,2020-01-03,2020-01-06,2020-01-07,2020-01-08,2020-01-09,2020-01-10,2020-01-13,2020-01-14,2020-01-15,2020-01-16,2020-01-17,2020-01-20,2020-01-21,2020-01-22,2020-01-23,2020-01-24,2020-01-27,2020-01-28,2020-01-29,2020-01-30,2020-01-31,2020-02-03,2020-02-04,2020-02-05,2020-02-06,2020-02-07,2020-02-10,2020-02-11,2020-02-12,2020-02-13,2020-02-14,2020-02-17,2020-02-18,2020-02-19,2020-02-20,2020-02-21,2020-02-24,2020-02-25,2020-02-26,2020-02-27,2020-02-28,2020-03-02,2020-03-03,2020-03-04,2020-03-05,2020-03-06,2020-03-09,2020-03-10,2020-03-11,2020-03-12,2020-03-13,2020-03-16,2020-03-17,2020-03-18,2020-03-19,2020-03-20,2020-03-23,2020-03-24,2020-03-25,2020-03-26,2020-03-27,2020-03-30,2020-03-31,2020-04-01,2020-04-02,2020-04-03,2020-04-06,2020-04-07,2020-04-08,2020-04-09,2020-04-10,2020-04-13,2020-04-14,2020-04-15,2020-04-16,2020-04-17,2020-04-20,2020-04-21,2020-04-22,...,2020-06-08,2020-06-09,2020-06-10,2020-06-11,2020-06-15,2020-06-16,2020-06-17,2020-06-18,2020-06-19,2020-06-22,2020-06-23,2020-06-24,2020-06-25,2020-06-26,2020-06-29,2020-06-30,2020-07-01,2020-07-02,2020-07-03,2020-07-06,2020-07-07,2020-07-08,2020-07-09,2020-07-10,2020-07-13,2020-07-14,2020-07-15,2020-07-16,2020-07-17,2020-07-20,2020-07-21,2020-07-22,2020-07-23,2020-07-24,2020-07-27,2020-07-28,2020-07-29,2020-07-30,2020-08-04,2020-08-05,2020-08-06,2020-08-07,2020-08-10,2020-08-11,2020-08-12,2020-08-13,2020-08-14,2020-08-17,2020-08-18,2020-08-19,2020-08-20,2020-08-21,2020-08-24,2020-08-25,2020-08-26,2020-08-27,2020-08-28,2020-08-31,2020-09-01,2020-09-02,2020-09-03,2020-09-04,2020-09-07,2020-09-08,2020-09-09,2020-09-10,2020-09-11,2020-09-14,2020-09-15,2020-09-16,2020-09-17,2020-09-1

In [40]:
ts_dax[["DE000A1EWWW0", "DE0008404005", "DE000BASF111", "DE000BAY0017"]].T.head()

DATE,2019-12-04,2019-12-05,2019-12-06,2019-12-09,2019-12-10,2019-12-11,2019-12-12,2019-12-13,2019-12-16,2019-12-17,2019-12-18,2019-12-19,2019-12-20,2019-12-23,2019-12-24,2019-12-25,2019-12-26,2019-12-27,2019-12-30,2019-12-31,2020-01-01,2020-01-02,2020-01-03,2020-01-06,2020-01-07,2020-01-08,2020-01-09,2020-01-10,2020-01-13,2020-01-14,2020-01-15,2020-01-16,2020-01-17,2020-01-20,2020-01-21,2020-01-22,2020-01-23,2020-01-24,2020-01-27,2020-01-28,2020-01-29,2020-01-30,2020-01-31,2020-02-03,2020-02-04,2020-02-05,2020-02-06,2020-02-07,2020-02-10,2020-02-11,2020-02-12,2020-02-13,2020-02-14,2020-02-17,2020-02-18,2020-02-19,2020-02-20,2020-02-21,2020-02-24,2020-02-25,2020-02-26,2020-02-27,2020-02-28,2020-03-02,2020-03-03,2020-03-04,2020-03-05,2020-03-06,2020-03-09,2020-03-10,2020-03-11,2020-03-12,2020-03-13,2020-03-16,2020-03-17,2020-03-18,2020-03-19,2020-03-20,2020-03-23,2020-03-24,2020-03-25,2020-03-26,2020-03-27,2020-03-30,2020-03-31,2020-04-01,2020-04-02,2020-04-03,2020-04-06,2020-04-07,2020-04-08,2020-04-09,2020-04-10,2020-04-13,2020-04-14,2020-04-15,2020-04-16,2020-04-17,2020-04-20,2020-04-21,...,2020-06-10,2020-06-11,2020-06-15,2020-06-16,2020-06-17,2020-06-18,2020-06-19,2020-06-22,2020-06-23,2020-06-24,2020-06-25,2020-06-26,2020-06-29,2020-06-30,2020-07-01,2020-07-02,2020-07-03,2020-07-06,2020-07-07,2020-07-08,2020-07-09,2020-07-10,2020-07-13,2020-07-14,2020-07-15,2020-07-16,2020-07-17,2020-07-20,2020-07-21,2020-07-22,2020-07-23,2020-07-24,2020-07-27,2020-07-28,2020-07-29,2020-07-30,2020-08-04,2020-08-05,2020-08-06,2020-08-07,2020-08-10,2020-08-11,2020-08-12,2020-08-13,2020-08-14,2020-08-17,2020-08-18,2020-08-19,2020-08-20,2020-08-21,2020-08-24,2020-08-25,2020-08-26,2020-08-27,2020-08-28,2020-08-31,2020-09-01,2020-09-02,2020-09-03,2020-09-04,2020-09-07,2020-09-08,2020-09-09,2020-09-10,2020-09-11,2020-09-14,2020-09-15,2020-09-16,2020-09-17,2020-09-18,2020-09-21,2020-09-22,2020-09-23,2020-09-24,2020-09-25,2020-09-28,2020-09-29,2020-09-30,2020-10-01,2020-10-02,2020-10-05,2020-10-06,2020-10-07,2020-10-08,2020-10-09,2020-10-12,2020-10-13,2020-10-14,2020-10-15,2020-10-16,2020-10-19,2020-10-20,2020-10-21,2020-10-22,2020-10-23,2020-10-26,2020-10-27,2020-10-28,2020-10-29,2020-10-30
DE000A1EWWW0,262.40,256.20,254.500,252.475,268.400,268.4000,272.850,279.60,282.35,282.50,284.1250,286.50,281.900,280.600,281.10,285.30,295.50,294.000,298.55,294.10,291.85,285.70,285.3750,283.300,284.05,279.15,280.200,278.00,278.00,275.550,269.000,268.350,270.65,274.4000,267.00,263.00,259.900,261.1500,263.0000,267.0000,261.55,263.700,264.05,270.2500,272.80,270.75,266.750,278.800,282.00,289.100,284.350,282.600,280.30,281.675,285.25,284.9000,293.55,297.30,299.000,297.95,296.900,300.90,300.00,298.50,301.45,301.20,299.45,301.950,303.95,304.90,306.700,304.8000,307.40,301.50,303.25,307.20,311.60,313.20,319.100,313.150,307.8000,308.85,313.050,310.55,316.00,336.25,322.75,317.95,306.800,308.80,309.50,309.00,312.00,317.70,312.30,308.50,305.25,305.450,311.90,315.85,...,297.05,299.15,301.95,298.40,294.55,294.05,296.80,288.75,284.90,286.55,284.15,287.700,280.60,258.80,259.60,252.05,259.85,256.85,260.60,258.85,262.30,265.95,266.100,270.30,271.50,269.10,277.20,277.90,277.55,277.60,280.45,282.35,280.00,276.75,271.30,276.55,270.75,266.20,265.00,263.70,261.70,261.80,279.80,282.00,278.00,281.00,287.00,285.00,287.50,294.50,298.90,300.00,290.10,282.80,280.20,271.50,279.20,289.50,290.20,293.00,288.20,292.40,288.70,290.60,291.40,295.20,288.80,290.90,295.30,297.70,292.050,292.300,279.80,283.900,278.50,278.60,274.60,275.000,278.50,270.90,262.000,265.60,270.70,276.70,270.25,278.00,285.10,285.30,286.20,295.20,281.900,289.80,286.80,288.200,292.300,295.10,288.50,291.70,289.600,295.40
DE0008404005,198.96,192.70,192.320,191.440,202.450,203.8000,203.575,203.25,202.75,204.75,205.2500,205.80,205.750,205.075,205.15,205.55,203.30,203.150,203.45,201.55,203.10,202.80,202.7000,201.150,200.95,201.20,201.300,198.54,199.00,198.640,199.240,199.080,198.54,198.6000,196.40,195.08,197.680,198.9400,198.5800,1